<!-- # Su Webtool — Version 2 refactored notebook

This is a safer refactor of the original interactive clinical decision-flow notebook. The aim is to make the architecture easier to understand without changing the intended widget behaviour.

**How to use:** run **Kernel → Restart & Run All**, then interact with the flowchart from the displayed buttons.

**What changed in V2:**

- The giant single code cell has been split into digestible, labelled sections.
- Clinical thresholds and external links have been collected into a central configuration section.
- Four overwritten duplicate function definitions have been removed or replaced with explicit cleanup comments.
- Existing event-handler order is preserved so the UI should behave like the original.
- Areas that still need cautious manual refactoring are called out in notes rather than silently changed.
 -->

<!-- ## Architecture map

```text
User clicks a branch button
        ↓
Event handler updates state
        ↓
Container children are rebuilt
        ↓
Visible flowchart branch changes
        ↓
restore_state() reconstructs selected branches when needed
```

The notebook is organised around five layers:

| Layer | Purpose |
|---|---|
| Configuration | Central clinical thresholds, links, and common layout values. |
| UI skeleton | Buttons, containers, CSS, and initial three-column shell. |
| State | A shared dictionary that records selected branches and visible boxes. |
| Branch logic | Functions that build Primary Prevention, Secondary Prevention, FH, CKD, diabetes, ACS, drug-interaction, Non-HDL, triglycerides, LDL-C, and atorvastatin branches. |
| Wiring | Button `.on_click(...)` calls and final `restore_state()`. |
 -->

<!-- ## 1. Imports

Loads the widget and display libraries used throughout the notebook. -->

In [1]:
# @title Default title text
import ipywidgets as widgets
from IPython.display import display, HTML
display(HTML("""
<style>
body, .widget-html-content, .oval-box {
    font-family: Arial, sans-serif !important;
}
</style>
"""))

<!-- ## 2. Central configuration

In [2]:
# Central configuration / clinical constants
# ------------------------------------------------------------
FH_REFERRAL_THRESHOLDS = {
    "TC": 7.5,
    "LDL": 4.9,
    "Non-HDL": 5.9,
}

FH_DIRECT_REFERRAL_THRESHOLDS = {
    "TC": 9.0,
    "LDL": 6.5,
    "Non-HDL": 7.5,
}

LDL_C_TREATMENT_THRESHOLDS = {
    "ICOSAPENT_MIN": 1.0,
    "ICOSAPENT_MAX": 2.6,
    "EZETIMIBE_MONOTHERAPY_MAX": 2.6,
    "INCLISIRAN_MIN": 2.6,
    "CV_EVENT_PROMPT_MIN": 3.5,
    "PCSK9_MIN": 4.0,
}

NON_HDL_TREATMENT_THRESHOLDS = {
    "SECONDARY_TARGET": 2.6,
    "TARGET_REDUCTION_FRACTION": 0.40,
}


NON_HDL_TARGET_TEXT = (
    f'Is Non-HDL cholesterol less than '
    f'{NON_HDL_TREATMENT_THRESHOLDS["SECONDARY_TARGET"]} mmol/L?'
)


TRIGLYCERIDE_THRESHOLDS = {
    "HIGH": 1.7,
}

QRISK_THRESHOLDS = {
    "OFFER_STATIN_PERCENT": 10,
}

CLINICAL_LINKS = {
    "QRISK3": "https://www.qrisk.org/",
    "NICE_GUIDANCE": "https://www.nice.org.uk/guidance",
    "NICE_NG238": "https://www.nice.org.uk/guidance/ng238",
    "NICE_TA733_INCLISIRAN": "https://www.nice.org.uk/guidance/ta733",
    "NICE_TA805_ICOSAPENT": "https://www.nice.org.uk/guidance/ta805",
    "SAMPSON_LDL_CALCULATOR": "https://www.medcentral.com/calculators/cardiology/sampson-equation-for-low-density-lipoprotein-ldl-c",
    "NEELI_STATIN_INTOLERANCE": "https://docs.google.com/gview?embedded=false&url=https%3A%2F%2Fntag.nhs.uk%2Fwp-content%2Fuploads%2F2025%2F12%2FNEELI-2025-version-061125-approved-Dec-2025.pdf",
}

<!-- ## 3. CSS and global page styling

Defines the notebook styling used by buttons, connector lines, info boxes, and oval recommendation boxes. -->

In [3]:
# Define the styles for the buttons, lines, and input table
styles = """
<style>


.fh-step {
    width: 100%;
    max-width: 400px;
    box-sizing: border-box;
    text-align: center;
    white-space: normal;
    overflow-wrap: break-word;
    word-break: normal;
    margin-left: auto;
    margin-right: auto;
}

.fh-info-box {
    width: 100%;
    max-width: 400px;
    box-sizing: border-box;
    text-align: center;
    white-space: normal;
    overflow-wrap: break-word;
    margin-left: auto;
    margin-right: auto;
}



.circle-button {
    width: 50px !important;
    height: 50px !important;
    min-width: 50px !important;
    min-height: 50px !important;
    border-radius: 50% !important;
    background-color: #eeeeee !important;
    color: black !important;
    border: none !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    cursor: pointer !important;
    font-size: 14px !important;
    box-sizing: border-box !important;
}

.circle-button.button-selected {
    background-color: yellow !important;
    box-shadow: 0 0 0 2px #66b3ff !important;
}

.horizontal-line {
    height: 2px !important;
    background-color: black !important;
    width: 20px !important;
}

.horizontal-line-container {
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
}


body {
    background-colour: #E3E3E3; /* Set the page background colour to grey using hex numbers */
}
.button-container {
    display: flex; justify-content: center; width: 100%;
}
.button-box {
    display: flex; flex-direction: column; align-items: center; padding: 10px; min-height: 100px;
}
button {
    border: 2px solid black; background-coloyr: #f0f0f0; padding: 20px 40px; border-radius: 25px; /* Oval shape */
    font-size: 18px; margin: 10px;
}
.vertical-line {
    width: 2px; 
    background-color: black; 
    height: 50px; 
    margin: 0 auto;
}
.horizontal-line-container {
    display: flex; align-items: center; justify-content: center; width: 100%;
}
.horizontal-line {
    height: 2px;
    background-color: black;
    width: 20px;
}
.circle-button {
    width: 50px; height: 50px; background-colour: #EEEEEE; border: 2px solid black; border-radius: 50%;
    display: flex; align-items: center; justify-content: center; cursor: pointer;
}
.circle-button.button-selected {
    border-colour: yellow; /* Change border colour to yellow when selected */
}
.option-container {
    margin-top: 10px; padding: 10px; border: 2px solid black; border-radius: 8px; background-colour: #f9f9f9; text-align: center;
}
.option-title {
    font-size: 16px; margin-bottom: 10px; font-weight: bold;
}
.option-table {
    width: 100%; border-collapse: collapse;
}
.option-table td {
    border: 1px solid #ddd; padding: 10px; cursor: pointer; text-align: center;
}
.option-table td.selected {
    background-coluor: lightgreen;
}
.option-table td:hover {
    background-coloir: #e0e0e0;
}
.info-box {
    margin-top: 10px; padding: 10px; border: 2px dashed black; border-radius: 8px; background-colour: #f9f9f9; text-align: left; /* Left justified text */
    width: 400px;
}
.button-selected {
    background-colour: yellow;
}
.expandable-section {
    margin-top: 10px; padding: 10px; border: 2px solid black; border-radius: 25px; background-coloir: #f9f9f9; text-align: center; cursor: pointer;
}
.expand-button {
    cursor: pointer; coluor: blue; text-decoration: underline;
}
.oval-box {
    padding: 20px 40px;
    border: 2px solid black;
    border-radius: 25px;
    background-colour: #f0f0f0; /* The usual grey */
    font-size: 18px;
    margin: 10px;
    text-align: center;
}

.large-widget-label label {
    font-size: 18px !important;
}

.qrisk3-input-row .widget-label {
    font-size: 16px !important;
    font-weight: 500 !important;
}

.qrisk3-input-row input {
    font-size: 15px !important;
    max-width: 70px !important;
}

.qrisk-info-box {
    text-align: center;
    padding: 12px;
    border: 2px solid #0066cc;
    border-radius: 10px;
    background-colour: #e8f4ff;
    margin: 10px;
}

.qrisk-link {
    display: inline-block !important;
    padding: 12px 24px !important;
    background-colour: #0066cc !important;
    colour: white !important;
    text-decoration: none !important;
    border-radius: 10px !important;
    font-weight: bold !important;
    font-size: 18px !important;

    animation: qriskGlow 1.8s ease-in-out infinite;
}

@keyframes qriskGlow {
    0% {
        box-shadow: 0 0 2px rgba(0,102,204,0.3);
    }

    50% {
        box-shadow: 0 0 18px rgba(0,102,204,1);
    }

    100% {
        box-shadow: 0 0 2px rgba(0,102,204,0.3);
    }
}


</style>
"""

# Initialize the HTML container for the styles
style_html = widgets.HTML(value=styles)

# ==========================================
# Pulse css
# ==========================================
pulse_css = widgets.HTML("""

<style>

.green-pulse-button,
.green-pulse-button .widget-button,
.green-pulse-button button {
    background-color: #4CAF50 !important;
    border: 2px solid #2E7D32 !important;
    color: white !important;
    font-weight: bold !important;
    animation: greenButtonPulse 1.8s ease-in-out infinite !important;
}

@keyframes greenButtonPulse {
    0% {
        box-shadow: 0 0 4px rgba(76,175,80,0.4);
        transform: scale(1);
    }

    50% {
        box-shadow: 0 0 20px rgba(76,175,80,1);
        transform: scale(1.05);
    }

    100% {
        box-shadow: 0 0 4px rgba(76,175,80,0.4);
        transform: scale(1);
    }
}

</style>













<style>
.gentle-yellow-pulse {
    background-colour: #FFD966 !important;
    colour: black !important;
    border: 2px solid #E0B800 !important;
    animation: gentlePulse 1.8s ease-in-out infinite;
}

@keyframes gentlePulse {
    0% {
        box-shadow: 0 0 2px rgba(255, 217, 102, 0.4);
        transform: scale(1.00);
    }
    50% {
        box-shadow: 0 0 12px rgba(255, 217, 102, 0.9);
        transform: scale(1.025);
    }
    100% {
        box-shadow: 0 0 2px rgba(255, 217, 102, 0.4);
        transform: scale(1.00);
    }
}

.qrisk-link {
    font-size: 20px;
    font-weight: bold;
    colour: #0066cc !important;
    text-decoration: underline;

    animation: qriskGlow 1.8s ease-in-out infinite;
}

@keyframes qriskGlow {
    0% {
        text-shadow: 0 0 2px rgba(0,102,204,0.3);
    }

    50% {
        text-shadow: 0 0 12px rgba(0,102,204,1);
    }

    100% {
        text-shadow: 0 0 2px rgba(0,102,204,0.3);
    }
}

@keyframes qriskPulse {
    0% {
        box-shadow: 0 0 2px rgba(0,102,204,0.4);
        transform: scale(1.0);
    }

    50% {
        box-shadow: 0 0 18px rgba(0,102,204,1);
        transform: scale(1.03);
    }

    100% {
        box-shadow: 0 0 2px rgba(0,102,204,0.4);
        transform: scale(1.0);
    }
}

.target-nonhdl-box {
    box-sizing: border-box;
    width: 100%;
    margin: 12px auto;
    padding: 16px 18px;
    border: 2px solid #ff9800;
    border-radius: 28px;
    background-color: #fff4e5;
    text-align: center;
    box-shadow: 0 0 8px rgba(255, 152, 0, 0.45);
    animation: targetNonHdlGlow 2.2s ease-in-out infinite;
    overflow: visible !important;
}

@keyframes targetNonHdlGlow {
    0% {
        box-shadow: 0 0 4px rgba(255, 152, 0, 0.35);
    }
    50% {
        box-shadow: 0 0 18px rgba(255, 152, 0, 0.85);
    }
    100% {
        box-shadow: 0 0 4px rgba(255, 152, 0, 0.35);
    }
}

.target-nonhdl-title {
    font-size: 15px;
    font-weight: bold;
    line-height: 1.45;
    margin-bottom: 12px;
    max-width: 400px;
}

.followup-green-box {
    box-sizing: border-box;
    width: 100%;
    margin: 12px auto;
    padding: 14px 18px;
    border: 2px solid #2e7d32;
    border-radius: 28px;
    background-color: #e8f5e9;
    text-align: center;
    font-weight: bold;
    font-size: 15px;
    color: #1b5e20;
    box-shadow: 0 0 8px rgba(76, 175, 80, 0.35);
    animation: followUpGlow 2.5s ease-in-out infinite;
}

@keyframes followUpGlow {
    0% {
        box-shadow: 0 0 5px rgba(76, 175, 80, 0.25);
    }
    50% {
        box-shadow: 0 0 14px rgba(76, 175, 80, 0.55);
    }
    100% {
        box-shadow: 0 0 5px rgba(76, 175, 80, 0.25);
    }
}

</style>
""")

display(pulse_css)


HTML(value='\n\n<style>\n@keyframes greenPulse {\n    0% {\n        box-shadow: 0 0 0px rgba(0, 200, 0, 0.0);\…

<!-- ## 4. Optional UI helper functions

Small helper factories for future cleanup. Existing widgets below remain mostly unchanged to preserve behaviour. -->

In [4]:
# Small optional UI helpers for future refactoring
# ------------------------------------------------------------

ROUND_BUTTON_LAYOUT = widgets.Layout(width='50px', height='50px', border_radius='50%')
OVAL_BUTTON_LAYOUT = widgets.Layout(width='180px', height='50px', border_radius='25px')


def make_round_button(label: str) -> widgets.Button:
    """Create a circular Yes/No/? style button."""
    return widgets.Button(description=label, layout=ROUND_BUTTON_LAYOUT)


def make_oval_button(label: str, width: str = '180px') -> widgets.Button:
    """Create a rounded flowchart button."""
    return widgets.Button(
        description=label,
        layout=widgets.Layout(width=width, height='50px', border_radius='25px'),
    )


<!-- ## 5. Top-level buttons and empty containers

Creates the main Primary Prevention, FH referral, and Secondary Prevention buttons, plus empty containers that later branch functions populate. -->

In [5]:
# Define the buttons
primary_button = widgets.Button(description="Primary Prevention", layout=widgets.Layout(width='180px', height='50px', border_radius='25px'))
hyperlipidaemia_button = widgets.Button(description="Refer for possible FH?", layout=widgets.Layout(width='220px', height='50px', border_radius='25px'))
secondary_button = widgets.Button(description="Secondary Prevention", layout=widgets.Layout(width='180px', height='50px', border_radius='25px'))

hyperlipidaemia_button.style.text_wrap = True  # Allows text wrapping
hyperlipidaemia_button.style.text_align = "center"  # Center the text

# Circular 'No', 'Yes', and '?' buttons
no_button = widgets.Button(description="No", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))
yes_button = widgets.Button(description="Yes", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))
info_button = widgets.Button(description="?", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))
ckd_no_button = widgets.Button(description="No", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))
ckd_yes_button = widgets.Button(description="Yes", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))
ckd_info_button = widgets.Button(description="?", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))
diabetes_no_button = widgets.Button(description="No", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))
diabetes_yes_button = widgets.Button(description="Yes", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))
diabetes_info_button = widgets.Button(description="?", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))
type1_button = widgets.Button(description="Type 1 diabetes?", layout=widgets.Layout(width='200px', height='50px', border_radius='25px'))
type2_button = widgets.Button(description="Type 2 diabetes?", layout=widgets.Layout(width='200px', height='50px', border_radius='25px'))

# Create containers for the primary button and its flowchart components
primary_flowchart_container = widgets.VBox([primary_button], layout=widgets.Layout(align_items='center'))
age_input_container = widgets.VBox()  # Container for age input table
info_box_container = widgets.VBox()  # Container for the info box
ckd_info_box_container = widgets.VBox()  # Container for the CKD info box
ckd_yes_section_container = widgets.VBox()  # Container for the CKD 'Yes' section
diabetes_question_container = widgets.VBox()  # Container for the diabetes question and options
diabetes_info_box_container = widgets.VBox()  # Container for the diabetes info box
diabetes_yes_section_container = widgets.VBox()  # Container for the diabetes 'Yes' section


# Question 3
secondary_flowchart_container = widgets.VBox([secondary_button], layout=widgets.Layout(align_items='center'))
secondary_yes_button = widgets.Button(description="Yes", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))
secondary_no_button = widgets.Button(description="No", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))
secondary_info_button = widgets.Button(description="?", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))
secondary_info_box_container = widgets.VBox()

# Containers for the ACS flowchart
acs_flowchart_container = widgets.VBox()
acs_info_box_container = widgets.VBox()  # Container for the ACS info box

# ACS Button Definitions
acs_yes_button = widgets.Button(description="Yes", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))
acs_no_button = widgets.Button(description="No", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))
acs_info_button = widgets.Button(description="?", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))

# Define buttons for the new drug interaction question
drug_interaction_yes_button = widgets.Button(description="Yes", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))
drug_interaction_no_button = widgets.Button(description="No", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))
drug_interaction_info_button = widgets.Button(description="?", layout=widgets.Layout(width='50px', height='50px', border_radius='50%'))

drug_interaction_question_html = widgets.VBox()
drug_interaction_ovals_container = widgets.VBox()
drug_interaction_info_box_container = widgets.VBox()


non_hdl_flowchart_container = widgets.VBox([], layout=widgets.Layout(align_items='center'))

# Container for the new question flow
drug_interaction_flowchart_container = widgets.VBox([
    drug_interaction_question_html,       # the fixed question branch
    drug_interaction_ovals_container,       # the dynamic ovals that update on yes/no
    non_hdl_flowchart_container             # the branch that follows
])




# LDL questions
ldl_yes_button = widgets.Button(
    description="Yes", 
    layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
)
ldl_info_button = widgets.Button(
    description="?", 
    layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
)
ldl_no_button = widgets.Button(
    description="No", 
    layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
)
ldl_flowchart_container = widgets.VBox([], layout=widgets.Layout(align_items='center'))


# Flowchart and buttons for Secondary Prevention Question 4
# Buttons and flowchart container for Non-HDL cholesterol
non_hdl_yes_button = widgets.Button(
    description="Yes", 
    layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
)
non_hdl_no_button = widgets.Button(
    description="No", 
    layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
)
non_hdl_info_button = widgets.Button(
    description="?", 
    layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
)
non_hdl_info_box_container = widgets.VBox()  # Container for the '?' info box


# Buttons for the Triglycerides question
triglycerides_yes_button = widgets.Button(
    description="Yes",
    layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
)
triglycerides_no_button = widgets.Button(
    description="No",
    layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
)
triglycerides_info_button = widgets.Button(
    description="?",
    layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
)
triglycerides_info_box_container = widgets.VBox()  # Container for the '?' info box
triglycerides_flowchart_container = widgets.VBox()  # Container for the flowchart elements specific to Triglycerides



# Define new buttons for the Atorvastatin question branch
atorvastatin_yes_button = widgets.Button(
    description="Yes", 
    layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
)
atorvastatin_no_button = widgets.Button(
    description="No", 
    layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
)
atorvastatin_info_button = widgets.Button(
    description="?", 
    layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
)


# Define new buttons for the Atorvastatin tolerance branch
atorvastatin_tolerance_yes_button = widgets.Button(
    description="Yes", 
    layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
)
atorvastatin_tolerance_no_button = widgets.Button(
    description="No", 
    layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
)
atorvastatin_tolerance_info_button = widgets.Button(
    description="?", 
    layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
)





# Create a container for the "Refer for possible FH?" button
fh_container = widgets.VBox(
    [hyperlipidaemia_button],
    layout=widgets.Layout(min_height='100px', align_items='center')
)


<!-- ## 6. Familial hypercholesterolaemia referral input panel

Builds the cholesterol input panel, thresholds, referral logic, and FH-related question flow. -->

In [3]:
THRESHOLDS = FH_REFERRAL_THRESHOLDS


FH_INPUT_BOX_WIDTH = '330px'
FH_LABEL_WIDTH = '190px'
FH_VALUE_WIDTH = '95px'
FH_PANEL_WIDTH = '350px'
FH_OUTPUT_WIDTH = '420px'




def build_qrisk3_gate(width='400px'):
    qrisk3_input = widgets.FloatText(
        value=0,
        step=0.1,
        description='',
        layout=widgets.Layout(width='80px')
    )

    result_box = widgets.VBox()

    enter_button = widgets.Button(
        description="Enter",
        layout=widgets.Layout(width='90px', height='30px')
    )

    def on_enter(b):
        result_box.children = []

        score = qrisk3_input.value

        if score is None or score < 0 or score > 100:
            result_box.children = [
                widgets.HTML("""
                <div style="color:#b00020; font-weight:bold; text-align:center;">
                    Invalid QRISK3. Please enter a valid percentage.
                </div>
                """)
            ]
            return

        if score < QRISK_THRESHOLDS["OFFER_STATIN_PERCENT"]:
            result_box.children = [
                widgets.HTML(f"""
                <div class="oval-box" style="
                    background-color:#f2f2f2;
                    font-size:15px;
                    line-height:1.5;
                    width:{width};
                    box-sizing:border-box;
                ">
                    <b>QRISK3 &lt; {QRISK_THRESHOLDS["OFFER_STATIN_PERCENT"]}%</b><br><br>
                    Routine primary prevention statin therapy is not automatically indicated.<br><br>
                    Discuss lifestyle modification and consider statin therapy if this is the patient's preference after shared decision making.
                </div>
                """)
            ]
            return

        target_non_hdl_box, follow_up_box = build_target_non_hdl_section(width=width)

        result_box.children = [
            widgets.HTML(f"""
            <div class="oval-box" style="
                background-color:#f2f2f2;
                font-size:15px;
                line-height:1.5;
                width:{width};
                box-sizing:border-box;
            ">
                <b>QRISK3 ≥ {QRISK_THRESHOLDS["OFFER_STATIN_PERCENT"]}%</b><br><br>
                Patient meets threshold for primary prevention statin therapy.<br><br>
                Discuss lifestyle modification, address modifiable risk factors, and offer Atorvastatin 20 mg once daily.
            </div>
            """),
            target_non_hdl_box,
            follow_up_box
        ]

    enter_button.on_click(on_enter)

    qrisk3_input_row = widgets.HBox(
        [
            widgets.HTML("<b>Enter QRISK3 result (%):</b>"),
            qrisk3_input,
            enter_button
        ],
        layout=widgets.Layout(
            justify_content='center',
            align_items='center',
            gap='8px',
            width='100%'
        )
    )

    qrisk3_gate = widgets.VBox(
        [
            widgets.HTML(f"""
            <div style="
                text-align:center;
                padding:12px;
                border:2px solid #0066cc;
                border-radius:10px;
                background-color:#e8f4ff;
                margin:14px auto;
                width:{width};
                box-sizing:border-box;
            ">
                <b>Calculate QRISK3 score</b><br><br>

                <a href="{CLINICAL_LINKS["QRISK3"]}"
                   target="_blank"
                   class="qrisk-link"
                   style="
                       display:inline-block;
                       padding:12px 24px;
                       background-color:#0066cc;
                       color:white !important;
                       text-decoration:none;
                       border-radius:10px;
                       font-weight:bold;
                       font-size:18px;
                       animation: qriskPulse 1.8s ease-in-out infinite;
                   ">
                   <span style="color:white !important;">
                       Open QRISK3 Calculator
                   </span>
                </a>
            </div>
            """),
            qrisk3_input_row,
            result_box
        ],
        layout=widgets.Layout(
            align_items='center',
            width=width,
            margin='12px auto'
        )
    )

    return qrisk3_gate
    




def build_fh_input_section():
    vertical_line_html = widgets.HTML('<div class="vertical-line"></div>')

    def fh_label(text):
        return widgets.HTML(
            value=f"""
            <div style="
                width:{FH_LABEL_WIDTH};
                text-align:right;
                font-size:14px;
                font-weight:bold;
                line-height:1.4;
            ">
                {text}
            </div>
            """,
            layout=widgets.Layout(width=FH_LABEL_WIDTH)
        )

    tc_input = widgets.BoundedFloatText(
        value=0,
        min=0,
        max=20,
        step=0.1,
        description='',
        layout=widgets.Layout(width=FH_VALUE_WIDTH)
    )

    ldl_input = widgets.BoundedFloatText(
        value=0,
        min=0,
        max=20,
        step=0.1,
        description='',
        layout=widgets.Layout(width=FH_VALUE_WIDTH)
    )

    non_hdl_input = widgets.BoundedFloatText(
        value=0,
        min=0,
        step=0.1,
        description='',
        layout=widgets.Layout(width=FH_VALUE_WIDTH)
    )

    
    
    fh_input_grid = widgets.GridBox(
        children=[
            fh_label('Total Cholesterol (mmol/L):'),
            tc_input,
    
            fh_label('LDL-C (mmol/L):'),
            ldl_input,
    
            fh_label('Non-HDL (mmol/L):'),
            non_hdl_input,
        ],
        layout=widgets.Layout(
            display='grid',
            grid_template_columns=f'{FH_LABEL_WIDTH} {FH_VALUE_WIDTH}',
            grid_gap='6px 8px',
            align_items='center',
            justify_content='center',
            width=FH_INPUT_BOX_WIDTH
        )
    )


    submit_button = widgets.Button(
        description="Submit",
        layout=widgets.Layout(width='150px', height='40px')
    )
    


    input_column = widgets.VBox(
        [
            fh_input_grid,
    
            widgets.HTML('<div style="height:10px;"></div>'),
    
            submit_button
        ],
        layout=widgets.Layout(
            align_items='center',
            width=FH_PANEL_WIDTH,
            border='2px solid black',
            padding='12px 18px',
            box_sizing='border-box',
            overflow='visible'
        )
    )



    
    fh_output_area = widgets.Output(
        layout=widgets.Layout(
            width=FH_OUTPUT_WIDTH,
            max_width=FH_OUTPUT_WIDTH,
            overflow='visible'
        )
    )



    fh_qrisk3_gate_container = widgets.VBox(
        [build_qrisk3_gate(width=FH_OUTPUT_WIDTH)],
        layout=widgets.Layout(
            display='none',
            align_items='center',
            width=FH_OUTPUT_WIDTH
        )
    )
    
    fh_qrisk3_gate_container.add_class("fh-qrisk3-gate-container")


    hide_fh_qrisk3_js = """
    document.querySelectorAll('.fh-qrisk3-gate-container').forEach(el => {
        el.style.display = 'none';
    });
    """

    show_fh_qrisk3_js = """
    document.querySelectorAll('.fh-qrisk3-gate-container').forEach(el => {
        el.style.display = 'flex';
    });
    """



    

    def on_submit_clicked(b):
        with fh_output_area:
            fh_output_area.clear_output()
            
            tc_val = tc_input.value
            ldl_val = ldl_input.value
            nhdl_val = non_hdl_input.value
    
            tc_high = tc_val >= THRESHOLDS["TC"]
            ldl_high = ldl_val >= THRESHOLDS["LDL"]
            nhdl_high = nhdl_val >= THRESHOLDS["Non-HDL"]
    
            if not (tc_high or ldl_high or nhdl_high):

                msg_html = f'''
                <div style="display:flex; flex-direction:column; align-items:center; width:100%; max-width:420px; overflow:visible; box-sizing:border-box;">

                    
                    <div class="oval-box" style="
                        width:420px;
                        box-sizing:border-box;
                        padding:14px 18px;
                        font-size:14px;
                        line-height:1.6;
                        text-align:center;
                        white-space:normal;
                        margin:14px auto 0 auto;
                        overflow:visible;
                    ">
                        <b>Will not meet Simon Broome criteria for Familial Hypercholesterolaemia.</b><br><br>
                        
                        Assess QRISK3 score and offer primary prevention if QRISK3 &gt;{QRISK_THRESHOLDS["OFFER_STATIN_PERCENT"]}% or if patient preference for statin therapy.
                    </div>
                </div>
                '''

            
  
            else:
                msg_html = f'''
                <div style="display:flex; flex-direction:column; align-items:center; width:100%; max-width:420px; overflow:visible; box-sizing:border-box;">
                    <div class="vertical-line"></div>
            
                    <div style="margin: 10px 0; text-align: center; font-size: 14px;">
                        Have secondary causes of high cholesterol been excluded?
                    </div>


<div class="horizontal-line-container" style="justify-content: center; overflow: visible;">
    <div class="circle-button fh-choice" style="border: none;" onclick="
        document.querySelectorAll('.fh-choice').forEach(btn => btn.classList.remove('button-selected'));
        document.querySelectorAll('.age-choice').forEach(btn => btn.classList.remove('button-selected'));
        document.querySelectorAll('.pregnancy-choice').forEach(btn => btn.classList.remove('button-selected'));
        document.querySelectorAll('.xanthoma-choice').forEach(btn => btn.classList.remove('button-selected'));
        document.querySelectorAll('.chd-choice').forEach(btn => btn.classList.remove('button-selected'));
        document.querySelectorAll('.chd-age-choice').forEach(btn => btn.classList.remove('button-selected'));
        document.querySelectorAll('.chd-second-choice').forEach(btn => btn.classList.remove('button-selected'));
        document.querySelectorAll('.chd-second-age-choice').forEach(btn => btn.classList.remove('button-selected'));
        document.querySelectorAll('.cholesterol-fh-choice').forEach(btn => btn.classList.remove('button-selected'));

        this.classList.add('button-selected');

        document.getElementById('fh-info-box').style.display = 'none';
        document.getElementById('secondary-causes-advice-box').style.display = 'none';

        document.getElementById('age-question').style.display = 'flex';
        document.getElementById('age-referral-box').style.display = 'none';

        document.getElementById('pregnancy-section').style.display = 'none';
        document.getElementById('pregnancy-referral-box').style.display = 'none';

        document.getElementById('xanthoma-question').style.display = 'none';
        document.getElementById('xanthoma-referral-box').style.display = 'none';

        document.getElementById('fh-chd-question').style.display = 'none';
        document.getElementById('fh-chd-info-box').style.display = 'none';
        document.getElementById('fh-chd-age-question').style.display = 'none';
        document.getElementById('fh-chd-referral-box').style.display = 'none';
        document.getElementById('chd-age-referral-box').style.display = 'none';

        document.getElementById('fh-chd-second-question').style.display = 'none';
        document.getElementById('chd-second-info-box').style.display = 'none';
        document.getElementById('fh-chd-second-age-question').style.display = 'none';
        document.getElementById('chd-second-referral-box').style.display = 'none';

        document.getElementById('fh-cholesterol-family-question').style.display = 'none';
        document.getElementById('cholesterol-fh-referral-box').style.display = 'none';
        document.getElementById('cholesterol-fh-negative-box').style.display = 'none';
        {hide_fh_qrisk3_js}
        








        
    ">Yes</div>

    <div class="horizontal-line"></div>

    <div class="circle-button" style="border: none;" onclick="
        const box = document.getElementById('fh-info-box');
        box.style.display = (box.style.display === 'block') ? 'none' : 'block';
    ">?</div>

    <div class="horizontal-line"></div>

    <div class="circle-button fh-choice" style="border: none;" onclick="
        document.querySelectorAll('.fh-choice').forEach(btn => btn.classList.remove('button-selected'));
        document.querySelectorAll('.age-choice').forEach(btn => btn.classList.remove('button-selected'));
        document.querySelectorAll('.pregnancy-choice').forEach(btn => btn.classList.remove('button-selected'));
        document.querySelectorAll('.xanthoma-choice').forEach(btn => btn.classList.remove('button-selected'));
        document.querySelectorAll('.chd-choice').forEach(btn => btn.classList.remove('button-selected'));
        document.querySelectorAll('.chd-age-choice').forEach(btn => btn.classList.remove('button-selected'));
        document.querySelectorAll('.chd-second-choice').forEach(btn => btn.classList.remove('button-selected'));
        document.querySelectorAll('.chd-second-age-choice').forEach(btn => btn.classList.remove('button-selected'));
        document.querySelectorAll('.cholesterol-fh-choice').forEach(btn => btn.classList.remove('button-selected'));

        this.classList.add('button-selected');

        document.getElementById('fh-info-box').style.display = 'none';
        document.getElementById('secondary-causes-advice-box').style.display = 'block';

        document.getElementById('age-question').style.display = 'none';
        document.getElementById('age-referral-box').style.display = 'none';

        document.getElementById('pregnancy-section').style.display = 'none';
        document.getElementById('pregnancy-referral-box').style.display = 'none';

        document.getElementById('xanthoma-question').style.display = 'none';
        document.getElementById('xanthoma-referral-box').style.display = 'none';

        document.getElementById('fh-chd-question').style.display = 'none';
        document.getElementById('fh-chd-info-box').style.display = 'none';
        document.getElementById('fh-chd-age-question').style.display = 'none';
        document.getElementById('fh-chd-referral-box').style.display = 'none';
        document.getElementById('chd-age-referral-box').style.display = 'none';

        document.getElementById('fh-chd-second-question').style.display = 'none';
        document.getElementById('chd-second-info-box').style.display = 'none';
        document.getElementById('fh-chd-second-age-question').style.display = 'none';
        document.getElementById('chd-second-referral-box').style.display = 'none';

        document.getElementById('fh-cholesterol-family-question').style.display = 'none';
        document.getElementById('cholesterol-fh-referral-box').style.display = 'none';
        document.getElementById('cholesterol-fh-negative-box').style.display = 'none';
        {hide_fh_qrisk3_js}
    ">No</div>
</div>

<!-- Info box for secondary causes -->
<div id="fh-info-box" class="info-box" style="
    display:none;
    margin:12px auto 0 auto;
    width:340px;
    box-sizing:border-box;
    text-align:left;
    font-size:13px;
    line-height:1.5;
">
    <strong>1. Check thyroid function</strong> (for hypothyroidism)<br>
    <strong>2. Check HbA1c</strong> (for uncontrolled diabetes)<br>
    <strong>3. Check liver function</strong><br>
    <strong>4. Check U+Es and urine ACR</strong> (for renal disease)<br>
    <strong>5. Check for medications that may contribute</strong>
</div>

<!-- Advice if secondary causes have NOT been excluded -->
<div id="secondary-causes-advice-box" class="oval-box" style="
    font-size: 14px;
    margin-top: 15px;
    background-color: #f2f2f2;
    display: none;
    width:300px;
    box-sizing:border-box;
    padding:14px 18px;
    line-height:1.5;
    text-align:center;
">
    Please exclude secondary causes of high cholesterol and treat underlying cause as appropriate.
</div>



<!-- Age question -->
<div id="age-question" style="
    display: none;
    flex-direction: column;
    align-items: center;
    margin-top: 20px;
    font-weight: normal;
">
    <div class="vertical-line"></div>

    <div style="
        margin: 10px 0;
        text-align: center;
        font-size: 14px;
        font-weight: normal;
    ">
        Is the person above the age of 30?
    </div>

    <div class="horizontal-line-container" style="justify-content: center;">
        <div class="circle-button age-choice" style="border: none; font-weight: normal;" onclick="
            document.querySelectorAll('.age-choice').forEach(btn => btn.classList.remove('button-selected'));
            document.querySelectorAll('.pregnancy-choice').forEach(btn => btn.classList.remove('button-selected'));
            document.querySelectorAll('.xanthoma-choice').forEach(btn => btn.classList.remove('button-selected'));
            document.querySelectorAll('.chd-choice').forEach(btn => btn.classList.remove('button-selected'));
            document.querySelectorAll('.chd-age-choice').forEach(btn => btn.classList.remove('button-selected'));
            document.querySelectorAll('.chd-second-choice').forEach(btn => btn.classList.remove('button-selected'));
            document.querySelectorAll('.chd-second-age-choice').forEach(btn => btn.classList.remove('button-selected'));
            document.querySelectorAll('.cholesterol-fh-choice').forEach(btn => btn.classList.remove('button-selected'));

            this.classList.add('button-selected');

            const tc = {tc_val};
            const ldl = {ldl_val};
            const nhdl = {nhdl_val};

            
            const showReferral = (
                tc > {FH_DIRECT_REFERRAL_THRESHOLDS["TC"]} ||
                ldl > {FH_DIRECT_REFERRAL_THRESHOLDS["LDL"]} ||
                nhdl > {FH_DIRECT_REFERRAL_THRESHOLDS["Non-HDL"]}
            );
            
            const showXanthoma = (
                !showReferral &&
                (
                    tc >= {FH_REFERRAL_THRESHOLDS["TC"]} ||
                    ldl >= {FH_REFERRAL_THRESHOLDS["LDL"]} ||
                    nhdl >= {FH_REFERRAL_THRESHOLDS["Non-HDL"]}
                )
            );

            document.getElementById('age-referral-box').style.display = 'none';

            document.getElementById('pregnancy-section').style.display = 'none';
            document.getElementById('pregnancy-referral-box').style.display = 'none';

            document.getElementById('xanthoma-question').style.display = 'none';
            document.getElementById('xanthoma-referral-box').style.display = 'none';

            document.getElementById('fh-chd-question').style.display = 'none';
            document.getElementById('fh-chd-info-box').style.display = 'none';

            document.getElementById('fh-chd-age-question').style.display = 'none';
            document.getElementById('fh-chd-referral-box').style.display = 'none';
            document.getElementById('chd-age-referral-box').style.display = 'none';

            document.getElementById('fh-chd-second-question').style.display = 'none';
            document.getElementById('chd-second-info-box').style.display = 'none';

            document.getElementById('fh-chd-second-age-question').style.display = 'none';
            document.getElementById('chd-second-referral-box').style.display = 'none';

            document.getElementById('fh-cholesterol-family-question').style.display = 'none';
            document.getElementById('cholesterol-fh-referral-box').style.display = 'none';
            document.getElementById('cholesterol-fh-negative-box').style.display = 'none';
            {hide_fh_qrisk3_js}




            

            if (showReferral) {{
                document.getElementById('age-referral-box').style.display = 'block';
            }} else if (showXanthoma) {{
                document.getElementById('xanthoma-question').style.display = 'flex';
            }}
        ">Yes</div>

        <div class="horizontal-line"></div>

        <div class="circle-button age-choice" style="border: none; font-weight: normal;" onclick="
            document.querySelectorAll('.age-choice').forEach(btn => btn.classList.remove('button-selected'));
            document.querySelectorAll('.pregnancy-choice').forEach(btn => btn.classList.remove('button-selected'));
            document.querySelectorAll('.xanthoma-choice').forEach(btn => btn.classList.remove('button-selected'));
            document.querySelectorAll('.chd-choice').forEach(btn => btn.classList.remove('button-selected'));
            document.querySelectorAll('.chd-age-choice').forEach(btn => btn.classList.remove('button-selected'));
            document.querySelectorAll('.chd-second-choice').forEach(btn => btn.classList.remove('button-selected'));
            document.querySelectorAll('.chd-second-age-choice').forEach(btn => btn.classList.remove('button-selected'));
            document.querySelectorAll('.cholesterol-fh-choice').forEach(btn => btn.classList.remove('button-selected'));

            this.classList.add('button-selected');

            document.getElementById('age-referral-box').style.display = 'none';

            document.getElementById('pregnancy-section').style.display = 'flex';
            document.getElementById('pregnancy-referral-box').style.display = 'none';
            document.getElementById('pregnancy-not-excluded-box').style.display = 'none';

            document.getElementById('xanthoma-question').style.display = 'none';
            document.getElementById('xanthoma-referral-box').style.display = 'none';

            document.getElementById('fh-chd-question').style.display = 'none';
            document.getElementById('fh-chd-info-box').style.display = 'none';

            document.getElementById('fh-chd-age-question').style.display = 'none';
            document.getElementById('fh-chd-referral-box').style.display = 'none';
            document.getElementById('chd-age-referral-box').style.display = 'none';

            document.getElementById('fh-chd-second-question').style.display = 'none';
            document.getElementById('chd-second-info-box').style.display = 'none';

            document.getElementById('fh-chd-second-age-question').style.display = 'none';
            document.getElementById('chd-second-referral-box').style.display = 'none';

            document.getElementById('fh-cholesterol-family-question').style.display = 'none';
            document.getElementById('cholesterol-fh-referral-box').style.display = 'none';
            document.getElementById('cholesterol-fh-negative-box').style.display = 'none';
        ">No</div>
    </div>

    <!-- Green referral box -->
    <div id="age-referral-box" class="oval-box" style="
        font-size: 14px;
        font-weight: normal;
        margin-top: 15px;
        background-color: lightgreen;
        display: none;
    ">
        Please refer to Lipid Clinic and include details of patient’s past medical history and family history of cardiovascular disease and cholesterol levels, if known.
    </div>



<!-- Pregnancy exclusion section if age <30 -->
<div id="pregnancy-section" style="
    display: none;
    flex-direction: column;
    align-items: center;
    margin-top: 20px;
    font-weight: normal;
">
    <div class="vertical-line"></div>

    <div class="oval-box" style="
        font-size: 14px;
        margin-top: 15px;
        background-color: #f2f2f2;
        width:350px;
        box-sizing:border-box;
        padding:14px 18px;
        line-height:1.5;
        text-align:center;
    ">
        Please exclude pregnancy as cause of high cholesterol in any females of childbearing age.
    </div>

    <div class="vertical-line"></div>

    <div style="
        margin: 10px 0;
        text-align: center;
        font-size: 14px;
        font-weight: normal;
    ">
        Pregnancy excluded?
    </div>

    <div class="horizontal-line-container" style="justify-content: center;">
        <div class="circle-button pregnancy-choice" style="border: none; font-weight: normal;" onclick="
            document.querySelectorAll('.pregnancy-choice').forEach(btn => btn.classList.remove('button-selected'));
            this.classList.add('button-selected');

            document.getElementById('pregnancy-referral-box').style.display = 'block';
            document.getElementById('pregnancy-not-excluded-box').style.display = 'none';
        ">Yes</div>

        <div class="horizontal-line"></div>

        <div class="circle-button pregnancy-choice" style="border: none; font-weight: normal;" onclick="
            document.querySelectorAll('.pregnancy-choice').forEach(btn => btn.classList.remove('button-selected'));
            this.classList.add('button-selected');

            document.getElementById('pregnancy-referral-box').style.display = 'none';
            document.getElementById('pregnancy-not-excluded-box').style.display = 'block';
        ">No</div>
    </div>

    <div id="pregnancy-referral-box" class="oval-box" style="
        font-size: 14px;
        margin-top: 15px;
        background-color: lightgreen;
        display: none;
    ">
        Please refer to local Lipid Clinic for assessment of possible familial hypercholesterolaemia
    </div>

    <div id="pregnancy-not-excluded-box" class="oval-box" style="
        font-size: 14px;
        margin-top: 15px;
        background-color: #f2f2f2;
        display: none;
        width:350px;
        box-sizing:border-box;
        padding:14px 18px;
        line-height:1.5;
        text-align:center;
    ">
        Please exclude pregnancy as a possible cause of high cholesterol before proceeding.
    </div>
</div>


                        
                        <!-- Xanthoma question -->
                        <div id="xanthoma-question" style="
                            display: none;
                            flex-direction: column;
                            align-items: center;
                            margin-top: 20px;
                            font-weight: normal;
                        ">
                            <div class="vertical-line"></div>
                        
                            <div class="fh-step" style="
                                margin: 10px 0;
                                font-size: 14px;
                                font-weight: normal;
                            ">
                                Are tendon xanthomata visible on patient or is there a family history of tendon xanthoma?
                            </div>
                        
                            <div class="horizontal-line-container" style="justify-content: center;">
                                <div class="circle-button xanthoma-choice" style="border: none; font-weight: normal;" onclick="
                                    document.querySelectorAll('.xanthoma-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.chd-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.chd-age-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.chd-second-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.chd-second-age-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.cholesterol-fh-choice').forEach(btn => btn.classList.remove('button-selected'));
                        
                                    this.classList.add('button-selected');
                        
                                    document.getElementById('xanthoma-referral-box').style.display = 'block';
                        
                                    document.getElementById('fh-chd-question').style.display = 'none';
                                    document.getElementById('fh-chd-info-box').style.display = 'none';
                        
                                    document.getElementById('fh-chd-age-question').style.display = 'none';
                                    document.getElementById('fh-chd-referral-box').style.display = 'none';
                                    document.getElementById('chd-age-referral-box').style.display = 'none';
                        
                                    document.getElementById('fh-chd-second-question').style.display = 'none';
                                    document.getElementById('chd-second-info-box').style.display = 'none';
                        
                                    document.getElementById('fh-chd-second-age-question').style.display = 'none';
                                    document.getElementById('chd-second-referral-box').style.display = 'none';
                        
                                    document.getElementById('fh-cholesterol-family-question').style.display = 'none';
                                    document.getElementById('cholesterol-fh-referral-box').style.display = 'none';
                                    document.getElementById('cholesterol-fh-negative-box').style.display = 'none';
                                    {hide_fh_qrisk3_js}
                                ">Yes</div>
                        
                                <div class="horizontal-line"></div>
                        
                                <div class="circle-button" style="border: none; font-weight: normal;" onclick="
                                    // no action
                                ">?</div>
                        
                                <div class="horizontal-line"></div>
                        
                                <div class="circle-button xanthoma-choice" style="border: none; font-weight: normal;" onclick="
                                    document.querySelectorAll('.xanthoma-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.chd-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.chd-age-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.chd-second-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.chd-second-age-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.cholesterol-fh-choice').forEach(btn => btn.classList.remove('button-selected'));
                        
                                    this.classList.add('button-selected');
                        
                                    document.getElementById('xanthoma-referral-box').style.display = 'none';
                        
                                    document.getElementById('fh-chd-question').style.display = 'flex';
                                    document.getElementById('fh-chd-info-box').style.display = 'none';
                        
                                    document.getElementById('fh-chd-age-question').style.display = 'none';
                                    document.getElementById('fh-chd-referral-box').style.display = 'none';
                                    document.getElementById('chd-age-referral-box').style.display = 'none';
                        
                                    document.getElementById('fh-chd-second-question').style.display = 'none';
                                    document.getElementById('chd-second-info-box').style.display = 'none';
                        
                                    document.getElementById('fh-chd-second-age-question').style.display = 'none';
                                    document.getElementById('chd-second-referral-box').style.display = 'none';
                        
                                    document.getElementById('fh-cholesterol-family-question').style.display = 'none';
                                    document.getElementById('cholesterol-fh-referral-box').style.display = 'none';
                                    document.getElementById('cholesterol-fh-negative-box').style.display = 'none';
                                    {hide_fh_qrisk3_js}
                                ">No</div>
                            </div>
                        
                            <!-- Advice box for xanthoma = Yes -->
                            <div id="xanthoma-referral-box" class="oval-box" style="font-size: 14px; margin-top: 15px; background-color: lightgreen; display: none;">
                                Please refer to local Lipid Clinic for assessment of possible familial hypercholesterolaemia
                            </div>
                        </div>
                        
                        

                        <!-- Next question if xanthoma = No -->
                        <div id="fh-chd-question" style="display: none; flex-direction: column; align-items: center; margin-top: 20px;">
                            <div class="vertical-line"></div>
                        
                            <div class="fh-step" style="
                                margin: 10px auto;
                                font-size: 14px;
                                display: block;
                            ">
                                Is there a family history of premature coronary heart disease (CHD) in a 1st-degree relative (parent, sibling, or child)?
                            </div>
                        
                            <div class="horizontal-line-container" style="justify-content: center;">




                            <div class="circle-button chd-choice" style="border: none;" onclick="
                                document.querySelectorAll('.chd-choice').forEach(btn => btn.classList.remove('button-selected'));
                                document.querySelectorAll('.chd-age-choice').forEach(btn => btn.classList.remove('button-selected'));
                                document.querySelectorAll('.chd-second-choice').forEach(btn => btn.classList.remove('button-selected'));
                                document.querySelectorAll('.chd-second-age-choice').forEach(btn => btn.classList.remove('button-selected'));
                                document.querySelectorAll('.cholesterol-fh-choice').forEach(btn => btn.classList.remove('button-selected'));
                            
                                this.classList.add('button-selected');
                            
                                document.getElementById('fh-chd-info-box').style.display = 'none';
                                document.getElementById('fh-chd-referral-box').style.display = 'none';
                                document.getElementById('chd-age-referral-box').style.display = 'none';
                            
                                document.getElementById('fh-chd-age-question').style.display = 'flex';
                            
                                document.getElementById('fh-chd-second-question').style.display = 'none';
                                document.getElementById('chd-second-info-box').style.display = 'none';
                                document.getElementById('fh-chd-second-age-question').style.display = 'none';
                                document.getElementById('chd-second-referral-box').style.display = 'none';
                            
                                document.getElementById('fh-cholesterol-family-question').style.display = 'none';
                                document.getElementById('cholesterol-fh-referral-box').style.display = 'none';
                                document.getElementById('cholesterol-fh-negative-box').style.display = 'none';
                                {hide_fh_qrisk3_js}
                            ">Yes</div>
                                                        




                        
                                <div class="horizontal-line"></div>
                        
                                <div class="circle-button" style="border: none;" onclick="
                                    const box = document.getElementById('fh-chd-info-box');
                                    box.style.display = (box.style.display === 'block') ? 'none' : 'block';
                                ">?</div>
                        
                                <div class="horizontal-line"></div>

                                <div class="circle-button chd-choice" style="border: none;" onclick="
                                    document.querySelectorAll('.chd-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.chd-age-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.chd-second-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.chd-second-age-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.cholesterol-fh-choice').forEach(btn => btn.classList.remove('button-selected'));
                                
                                    this.classList.add('button-selected');
                                
                                    document.getElementById('fh-chd-info-box').style.display = 'none';
                                
                                    document.getElementById('fh-chd-age-question').style.display = 'none';
                                    document.getElementById('fh-chd-referral-box').style.display = 'none';
                                    document.getElementById('chd-age-referral-box').style.display = 'none';
                                
                                    document.getElementById('fh-chd-second-question').style.display = 'flex';
                                    document.getElementById('chd-second-info-box').style.display = 'none';
                                
                                    document.getElementById('fh-chd-second-age-question').style.display = 'none';
                                    document.getElementById('chd-second-referral-box').style.display = 'none';
                                
                                    document.getElementById('fh-cholesterol-family-question').style.display = 'none';
                                    document.getElementById('cholesterol-fh-referral-box').style.display = 'none';
                                    document.getElementById('cholesterol-fh-negative-box').style.display = 'none';
                                ">No</div>
                                
                            </div>
                        
                            <div id="fh-chd-info-box" class="fh-info-box" style="
                                display:none;
                                margin:12px auto 0 auto;
                                box-sizing:border-box;
                                text-align:center;
                                font-size:13px;
                                line-height:1.5;
                                border:2px dashed black;
                                border-radius:28px;
                                padding:10px 16px;
                            ">
                                (i.e. MI, Coronary Artery Bypass Graft, PCI or definite coronary artery disease on coronary angiography)<br>
                                in a <b>first-degree</b> relative (parent, sibling or child)?
                            </div>
                        </div>




                        <!-- Referral advice for CHD first-degree -->
                        <div id="fh-chd-referral-box" class="oval-box" style="font-size: 14px; margin-top: 15px; background-color: lightgreen; display: none;">
                            . Please refer to local Lipid Clinic for assessment of possible familial hypercholesterolaemia
                        </div>

             
                        
                        <!-- CHD Age Check -->
                        <div id="fh-chd-age-question" style="display: none; flex-direction: column; align-items: center; margin-top: 20px;">
                            <div class="vertical-line"></div>
                        
                            <div style="margin: 10px 0; text-align: center; font-size: 14px;">
                                Was the first-degree relative below the age of 60 years when they were diagnosed with coronary heart disease?
                            </div>
                        
                            <div class="horizontal-line-container" style="justify-content: center;">
                                <div class="circle-button chd-age-choice" style="border: none;" onclick="
                                    document.querySelectorAll('.chd-age-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.chd-second-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.chd-second-age-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.cholesterol-fh-choice').forEach(btn => btn.classList.remove('button-selected'));
                        
                                    this.classList.add('button-selected');
                        
                                    document.getElementById('chd-age-referral-box').style.display = 'block';
                        
                                    document.getElementById('fh-chd-second-question').style.display = 'none';
                                    document.getElementById('chd-second-info-box').style.display = 'none';
                        
                                    document.getElementById('fh-chd-second-age-question').style.display = 'none';
                                    document.getElementById('chd-second-referral-box').style.display = 'none';
                        
                                    document.getElementById('fh-cholesterol-family-question').style.display = 'none';
                                    document.getElementById('cholesterol-fh-referral-box').style.display = 'none';
                                    document.getElementById('cholesterol-fh-negative-box').style.display = 'none';
                                    {hide_fh_qrisk3_js}
                                ">Yes</div>
                        
                                <div class="horizontal-line"></div>
                        
                                <div class="circle-button chd-age-choice" style="border: none;" onclick="
                                    document.querySelectorAll('.chd-age-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.chd-second-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.chd-second-age-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.cholesterol-fh-choice').forEach(btn => btn.classList.remove('button-selected'));
                        
                                    this.classList.add('button-selected');
                        
                                    document.getElementById('chd-age-referral-box').style.display = 'none';
                        
                                    document.getElementById('fh-chd-second-question').style.display = 'flex';
                                    document.getElementById('chd-second-info-box').style.display = 'none';
                        
                                    document.getElementById('fh-chd-second-age-question').style.display = 'none';
                                    document.getElementById('chd-second-referral-box').style.display = 'none';
                        
                                    document.getElementById('fh-cholesterol-family-question').style.display = 'none';
                                    document.getElementById('cholesterol-fh-referral-box').style.display = 'none';
                                    document.getElementById('cholesterol-fh-negative-box').style.display = 'none';
                                    {hide_fh_qrisk3_js}
                                ">No</div>
                            </div>
                        </div>
                        
                        
                        <!-- Referral advice after CHD age <60 -->
                        <div id="chd-age-referral-box" class="oval-box" style="font-size: 14px; margin-top: 15px; background-color: lightgreen; display: none;">
                            Please refer to local Lipid Clinic for assessment of possible familial hypercholesterolaemia
                        </div>
                        
                        
                        <!-- CHD second-degree relative question -->
                        <div id="fh-chd-second-question" style="display: none; flex-direction: column; align-items: center; margin-top: 20px;">
                            <div class="vertical-line"></div>
                        
                            <div class="fh-step" style="margin: 10px 0; font-size: 14px;">
                                Is there a family history of premature coronary heart disease (CHD) in a 2nd-degree relative (grandparent, uncle, auntie, grandchild, niece, or nep)
                            </div>
                        
                            <div class="horizontal-line-container" style="justify-content: center;">
                                <div class="circle-button chd-second-choice" style="border: none;" onclick="
                                    document.querySelectorAll('.chd-second-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.chd-second-age-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.cholesterol-fh-choice').forEach(btn => btn.classList.remove('button-selected'));
                        
                                    this.classList.add('button-selected');
                        
                                    document.getElementById('chd-second-info-box').style.display = 'none';
                        
                                    document.getElementById('fh-chd-second-age-question').style.display = 'flex';
                                    document.getElementById('chd-second-referral-box').style.display = 'none';
                        
                                    document.getElementById('fh-cholesterol-family-question').style.display = 'none';
                                    document.getElementById('cholesterol-fh-referral-box').style.display = 'none';
                                    document.getElementById('cholesterol-fh-negative-box').style.display = 'none';
                                    {hide_fh_qrisk3_js}
                                ">Yes</div>
                        
                                <div class="horizontal-line"></div>
                        
                                <div class="circle-button" style="border: none;" onclick="
                                    const box = document.getElementById('chd-second-info-box');
                                    box.style.display = (box.style.display === 'block') ? 'none' : 'block';
                                ">?</div>
                        
                                <div class="horizontal-line"></div>
                        
                                <div class="circle-button chd-second-choice" style="border: none;" onclick="
                                    document.querySelectorAll('.chd-second-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.chd-second-age-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.cholesterol-fh-choice').forEach(btn => btn.classList.remove('button-selected'));
                        
                                    this.classList.add('button-selected');
                        
                                    document.getElementById('chd-second-info-box').style.display = 'none';
                        
                                    document.getElementById('fh-chd-second-age-question').style.display = 'none';
                                    document.getElementById('chd-second-referral-box').style.display = 'none';
                        
                                    document.getElementById('fh-cholesterol-family-question').style.display = 'flex';
                                    document.getElementById('cholesterol-fh-referral-box').style.display = 'none';
                                    document.getElementById('cholesterol-fh-negative-box').style.display = 'none';
                                    {hide_fh_qrisk3_js}
                                ">No</div>
                            </div>
                        
                            <div id="chd-second-info-box" style="
                                display:none;
                                margin:12px auto 0 auto;
                                width:400px;
                                box-sizing:border-box;
                                text-align:center;
                                font-size:13px;
                                line-height:1.5;
                                border:2px dashed black;
                                border-radius:28px;
                                padding:10px 16px;
                            ">
                                (i.e. MI, Coronary Artery Bypass Graft, PCI or definite coronary artery disease on coronary angiography)
                            </div>
                        </div>


                        <!-- CHD second-degree age check -->
                        <div id="fh-chd-second-age-question" style="display: none; flex-direction: column; align-items: center; margin-top: 20px;">
                            <div class="vertical-line"></div>
                        
                            <div style="margin: 10px 0; text-align: center; font-size: 14px;">
                                Was the second-degree relative below the age of 50 years when they experienced coronary heart disease?
                            </div>
                        
                            <div class="horizontal-line-container" style="justify-content: center;">
                                <div class="circle-button chd-second-age-choice" style="border: none;" onclick="
                                    document.querySelectorAll('.chd-second-age-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.cholesterol-fh-choice').forEach(btn => btn.classList.remove('button-selected'));
                        
                                    this.classList.add('button-selected');
                        
                                    document.getElementById('chd-second-referral-box').style.display = 'block';
                        
                                    document.getElementById('fh-cholesterol-family-question').style.display = 'none';
                                    document.getElementById('cholesterol-fh-referral-box').style.display = 'none';
                                    document.getElementById('cholesterol-fh-negative-box').style.display = 'none';
                                    {hide_fh_qrisk3_js}
                                ">Yes</div>
                        
                                <div class="horizontal-line"></div>
                        
                                <div class="circle-button chd-second-age-choice" style="border: none;" onclick="
                                    document.querySelectorAll('.chd-second-age-choice').forEach(btn => btn.classList.remove('button-selected'));
                                    document.querySelectorAll('.cholesterol-fh-choice').forEach(btn => btn.classList.remove('button-selected'));
                        
                                    this.classList.add('button-selected');
                        
                                    document.getElementById('chd-second-referral-box').style.display = 'none';
                        
                                    document.getElementById('fh-cholesterol-family-question').style.display = 'flex';
                                    document.getElementById('cholesterol-fh-referral-box').style.display = 'none';
                                    document.getElementById('cholesterol-fh-negative-box').style.display = 'none';
                                    {hide_fh_qrisk3_js}
                                ">No</div>
                            </div>
                        </div>
                        
                        
                        <!-- Referral box after CHD in second-degree under age 50 -->
                        <div id="chd-second-referral-box" class="oval-box" style="font-size: 14px; margin-top: 15px; background-color: lightgreen; display: none;">
                            Please refer to local Lipid Clinic for assessment of possible familial hypercholesterolaemia
                        </div>
                        
                        
                        <!-- Cholesterol family history question -->
                        <div id="fh-cholesterol-family-question" style="display: none; flex-direction: column; align-items: center; margin-top: 20px;">
                            <div class="vertical-line"></div>
                        
                            <div style="margin: 10px 0; text-align: center; font-size: 14px;">
                                Is there a family history of total cholesterol &gt; {FH_REFERRAL_THRESHOLDS["TC"]} mmol/L
                                or LDL-cholesterol &gt; {FH_REFERRAL_THRESHOLDS["LDL"]} mmol/L<br>
                                in a <span style="font-weight: bold;">first-degree OR second-degree</span> relative?
                            </div>
                        
                            <div class="horizontal-line-container" style="justify-content: center;">
                                <div class="circle-button cholesterol-fh-choice" style="border: none;" onclick="
                                    document.querySelectorAll('.cholesterol-fh-choice').forEach(btn => btn.classList.remove('button-selected'));
                        
                                    this.classList.add('button-selected');
                        
                                    document.getElementById('cholesterol-fh-referral-box').style.display = 'block';
                                    document.getElementById('cholesterol-fh-negative-box').style.display = 'none';
                                    {hide_fh_qrisk3_js}
                                ">Yes</div>
                        
                                <div class="horizontal-line"></div>
                        
                                <div class="circle-button cholesterol-fh-choice" style="border: none;" onclick="
                                    document.querySelectorAll('.cholesterol-fh-choice').forEach(btn => btn.classList.remove('button-selected'));
                        
                                    this.classList.add('button-selected');
                        
                                    document.getElementById('cholesterol-fh-referral-box').style.display = 'none';
                                    document.getElementById('cholesterol-fh-negative-box').style.display = 'block';
                                    {show_fh_qrisk3_js}
                                ">No</div>
                            </div>
                        </div>
                        
                        
                        <!-- Final referral box -->
                        <div id="cholesterol-fh-referral-box" class="oval-box" style="font-size: 14px; margin-top: 15px; background-color: lightgreen; display: none;">
                            Please refer to local Lipid Clinic for assessment of possible familial hypercholesterolaemia
                        </div>
                        
                        
                        <!-- Final negative advice box -->
                        <div id="cholesterol-fh-negative-box" class="oval-box" style="
                            font-size: 14px;
                            margin-top: 15px;
                            background-color: #f2f2f2;
                            display: none;
                            width:400px;
                            box-sizing:border-box;
                            padding:14px 18px;
                            line-height:1.5;
                            text-align:center;
                        ">
                            Does not fulfil Simon Broome criteria for familial hypercholesterolaemia.<br><br>

                            Please perform QRISK3 score and offer statin for primary prevention if QRISK3 &gt; {QRISK_THRESHOLDS["OFFER_STATIN_PERCENT"]}% or patient preference for statin therapy.<br><br>                            
                            If there is still strong clinical suspicion to suggest familial hypercholesterolaemia please write Advice and Guidance request to local Lipid Clinic.
                            
                        </div>
                        
                            </div>
                        </div>
                        
                    </div>
                </div>
                '''





            display(HTML(msg_html))

    submit_button.on_click(on_submit_clicked)

    fh_inner_column = widgets.VBox(
        [
            input_column,
            fh_output_area,
            fh_qrisk3_gate_container
        ],
        layout=widgets.Layout(
            width=FH_OUTPUT_WIDTH,
            max_width=FH_OUTPUT_WIDTH,
            align_items='center',
            overflow='visible'
        )
    )

    return widgets.VBox(
        [
            vertical_line_html,
            widgets.HTML('<div style="height:5px;"></div>'),
            fh_inner_column
        ],
        layout=widgets.Layout(
            width=FH_OUTPUT_WIDTH,
            max_width=FH_OUTPUT_WIDTH,
            align_items='center',
            overflow='visible',
            flex='0 0 auto'
        )
    )        

    

NameError: name 'FH_REFERRAL_THRESHOLDS' is not defined

<!-- ## 7. Initial three-column display shell

Displays the three main entry buttons and inserts the CSS into the notebook output. -->

In [7]:
# APP_ZOOM = 0.5  # try 0.80 if opened branches still feel too wide


In [8]:
# Single-screen layout: match the patched v2 behaviour.
# The v3 version forced a 1120px-wide scroll shell, which is why the
# Secondary Prevention branch is pushed off-screen.
single_screen_css = widgets.HTML("""
<style>
/* Keep notebook output itself from adding an unnecessary local scrollbar */
.jp-OutputArea-output {
    overflow: visible !important;
}

/* Compact three-branch row */
.single-screen-flowchart {
    width: 100% !important;
    max-width: 100% !important;
    display: flex !important;
    flex-direction: row !important;
    flex-wrap: nowrap !important;
    justify-content: center !important;
    align-items: flex-start !important;
    gap: 28px !important;
    overflow: visible !important;
    box-sizing: border-box !important;
}

/* Let each branch size naturally instead of forcing a wide minimum */
.single-screen-flowchart .flowchart-branch {
    flex: 0 1 auto !important;
    min-width: 0 !important;
    width: auto !important;
    max-width: none !important;
    overflow: visible !important;
    box-sizing: border-box !important;
}

.single-screen-flowchart .widget-vbox,
.single-screen-flowchart .widget-hbox,
.single-screen-flowchart .widget-box {
    overflow: visible !important;
}
</style>
""")

for branch in (primary_flowchart_container, fh_container, secondary_flowchart_container):
    branch.layout = widgets.Layout(
        width='auto',
        min_width='0',
        align_items='center',
        overflow='visible',
        flex='0 1 auto'
    )
    branch.add_class('flowchart-branch')

button_container = widgets.HBox(
    [
        primary_flowchart_container,
        fh_container,
        secondary_flowchart_container
    ],
    layout=widgets.Layout(
        width='100%',
        max_width='100%',
        justify_content='center',
        align_items='flex-start',
        overflow='visible',
        flex_flow='row nowrap'
    )
)
button_container.add_class('single-screen-flowchart')

display(style_html, pulse_css, single_screen_css, button_container)


HTML(value='\n<style>\n\n\n.circle-button {\n    width: 50px !important;\n    height: 50px !important;\n    mi…

HTML(value='\n\n<style>\n@keyframes greenPulse {\n    0% {\n        box-shadow: 0 0 0px rgba(0, 200, 0, 0.0);\…

HTML(value='\n<style>\n/* Keep notebook output itself from adding an unnecessary local scrollbar */\n.jp-Outpu…

<!-- ## 8. Shared state dictionary

Stores which pathway, button, info box, and branch is currently active or visible. -->

In [9]:
state = {
    'primary_prevention': False,
    'no_button': False,
    'yes_button': False,
    'selected_age_option': None,
    'selected_button': None,
    'flowchart_created': False,
    'ckd_flowchart_created': False,
    'diabetes_flowchart_created': False,
    'diabetes_buttons_created': False,
    # 'current_state': None,
    'info_box_visible': False,
    'ckd_info_box_visible': False,
    'oval_box_visible': False,
    'ckd_yes_section_visible': False,
    'diabetes_question_visible': False,
    'diabetes_yes_section_visible': False,
    'diabetes_options_visible': False,
    'type1_options_visible': False,
    'type1_selected': False,
    'type2_selected': False,
    'type1_any_yes': False,  # Track if any Type 1 diabetes radio button is Yes
    'ckd_state': None,  # Track the state when CKD is selected
    'age_option_84_state': None,  # Track state for age option ≤ 84 years
    'type1_state': None,  # Track state for Type 1 diabetes options
    'secondary_prevention': None,

    # New ACS-specific states
    'acs_flowchart_created': False,  # Track if the ACS flowchart has been created
    'secondary_prevention_visible': False,
    'acs_yes_button': False,  # Track if the ACS Yes button has been selected
    'acs_oval_box_visible': False, # Track if ACS oval box is visible
    'acs_no_button': False,  # Track if the ACS No button has been selected
    'acs_info_box_visible': False,  # Track if the ACS info box is visible
    'secondary_yes_button': False,  # Track if the Secondary Prevention Yes button is selected
    'secondary_no_button': False,  # Track if the Secondary Prevention No button is selected
    'statin_oval_box_visible': False, # 

    # Drug interaction question specific states
    'drug_interaction_flowchart_created': False,
    'drug_interaction_yes_button': False,  # Track if the Drug Interaction Yes button is selected
    'drug_interaction_no_button': False,  # Track if the Drug Interaction No button is selected
    'drug_interaction_info_box_visible': False,  # Track if the Drug Interaction info box is visible
    'drug_interaction_oval_visible': False, 
    'drug_interaction_no_oval_visible': False,



    'non_hdl_flowchart_created': False,  # Tracks if Non-HDL cholesterol flowchart is created
    'non_hdl_yes_button': False,        # Tracks if the "Yes" button is clicked
    'non_hdl_no_button': False,         # Tracks if the "No" button is clicked
    'non_hdl_info_box_visible': False,  # Tracks visibility of the '?' info box    


    # Triglycerides Flowchart States
    'triglycerides_flowchart_created': False,  # Tracks if the Triglycerides flowchart has been created
    'triglycerides_yes_button': False,         # Tracks if the "Yes" button is selected for Triglycerides
    'triglycerides_no_button': False,          # Tracks if the "No" button is selected for Triglycerides
    'triglycerides_info_box_visible': False,   # Tracks visibility of the '?' info box for Triglycerides

    
    'current_state': None
}


<!-- ## 9. Small shared helpers and information toggles

Generic helpers for styling selected buttons and toggling explanatory information boxes. -->

In [10]:
# Function to update age option button styles
def update_age_option_styles(selected_button):
    age_option_1.style.button_color = '#EEEEEE'
    age_option_2.style.button_color = '#EEEEEE'
    selected_button.style.button_color = 'lightgreen'

# Function to toggle the info box
def toggle_info_box(b):
    state['info_box_visible'] = not state['info_box_visible']
    if state['info_box_visible']:
        info_box = widgets.HTML('''
        <div class="info-box">
            <strong>a. Coronary heart Disease</strong> (including Acute Coronary Syndrome, Myocardial Infarction, Unstable Angina or Coronary Revascularisation)<br>
            <strong>b. Peripheral Vascular Disease</strong><br>
            <strong>c. Ischaemic Stroke</strong>
        </div>
        ''')
        info_box_container.children = [info_box]
    else:
        info_box_container.children = []

# Function to toggle the CKD info box
def toggle_ckd_info_box(b):
    state['ckd_info_box_visible'] = not state['ckd_info_box_visible']
    if state['ckd_info_box_visible']:
        ckd_info_box = widgets.HTML('''
        <div class="info-box">
            eGFR < 60ml/min/1.73m<sup>2</sup> and/or albuminuria
        </div>
        ''')
        ckd_info_box_container.children = [ckd_info_box]
    else:
        ckd_info_box_container.children = []

# Function to toggle the diabetes info box
def toggle_diabetes_info_box(b):
    state['diabetes_info_box_visible'] = not state['diabetes_info_box_visible']
    if state['diabetes_info_box_visible']:
        diabetes_info_box = widgets.HTML('''
        <div class="info-box">
            Information about diabetes and related risks.
        </div>
        ''')
        diabetes_info_box_container.children = [diabetes_info_box]
    else:
        diabetes_info_box_container.children = []

# Function to highlight buttons
def highlight_button(buttons, selected_button):
    for btn in buttons:
        btn.style.button_color = '#EEEEEE'
    selected_button.style.button_color = 'yellow'

# Event handler for the 'Primary Prevention' button click


<!-- ## 10. Primary Prevention entry and age/CVD routing

Controls the Primary Prevention button, the first Yes/No question, and age routing. -->

In [11]:
def on_primary_button_click(b):

    
    if state['primary_prevention_visible']:
        # If already visible, hide the flowchart
        primary_flowchart_container.children = [primary_button]  # Reset to only show the primary button
        state['primary_prevention_visible'] = False
    else:
        # If not visible, show the flowchart
        state['current_state'] = 'primary_prevention'
        state['primary_prevention_visible'] = True
        state['no_button'] = False
        state['yes_button'] = False
        update_primary_flowchart()
    if b is not None:
        stop_secondary_to_primary_glow()


# Initialize the visibility state for primary prevention
state['primary_prevention_visible'] = False

def update_primary_flowchart():
    # Create the vertical line, question, horizontal line, and circle buttons as HTML widgets
    line_html = widgets.HTML('<div class="vertical-line"></div>')

    # Create a container for the horizontal line and circle buttons
    horizontal_line_container = widgets.HBox([
        yes_button,
        widgets.HTML('<div class="horizontal-line" style="flex-grow: 1;"></div>'),  # Add flex-grow to expand line
        info_button,
        widgets.HTML('<div class="horizontal-line" style="flex-grow: 1;"></div>'),  # Add flex-grow to expand line
        no_button
    ], layout=widgets.Layout(justify_content='center', align_items='center', width='100%'))

    # Create the question and add the horizontal line container below it
    question_html = widgets.VBox([
        widgets.HTML('<div style="font-size: 14px; text-align: center;">Does this person have Established cardiovascular disease?</div>'),
        horizontal_line_container,
        info_box_container
    ], layout=widgets.Layout(align_items='center'))

    # Update the container to include the line, question, and circle buttons
    primary_flowchart_container.children = [primary_button, line_html, question_html, age_input_container]

    if state['no_button']:
        on_no_button_click(None, restore=True)
    elif state['yes_button']:
        on_yes_button_click(None, restore=True)

state['primary_prevention_visible'] = False

# Event handler for the 'No' button click
# [V2 cleanup] Removed first overwritten on_no_button_click definition; later definition is the active one.
# Event handler for the 'Yes' button click


def start_primary_to_secondary_glow():
    yes_button.add_class("gentle-yellow-pulse")
    secondary_button.add_class("gentle-yellow-pulse")


def stop_primary_to_secondary_glow():
    yes_button.remove_class("gentle-yellow-pulse")
    secondary_button.remove_class("gentle-yellow-pulse")

def start_secondary_to_primary_glow():
    secondary_no_button.add_class("gentle-yellow-pulse")
    primary_button.add_class("gentle-yellow-pulse")

def stop_secondary_to_primary_glow():
    secondary_no_button.remove_class("gentle-yellow-pulse")
    primary_button.remove_class("gentle-yellow-pulse")
    
def on_no_button_click(b, restore=False):
    state['no_button'] = True
    state['yes_button'] = False
    stop_primary_to_secondary_glow()

    if not restore:
        state['current_state'] = 'no_button'

    # Create the HTML for the age question without a box and bold styling
    age_question_container = widgets.HTML('<div style="font-size: 14px; text-align: center; margin-top: 10px;">What is the person\'s age?</div>')
    global age_option_1, age_option_2
    age_option_1 = widgets.Button(description="≤ 84 years", layout=widgets.Layout(width='100px', height='50px'))
    age_option_2 = widgets.Button(description="> 84 years", layout=widgets.Layout(width='100px', height='50px'))

    # Add event handlers for the age options
    age_option_1.on_click(select_age_option_84)
    age_option_2.on_click(select_age_option_above_84)

    # Update the container for age input
    age_input_container.children = [
        age_question_container,
        widgets.HBox([age_option_1, age_option_2], layout=widgets.Layout(justify_content='center'))
    ]

    # Highlight 'No' button
    highlight_button([no_button, yes_button], no_button)
    state['selected_button'] = 'no_button'

    # Hide the Secondary Prevention branch if visible
    if state['secondary_prevention_visible']:
        secondary_flowchart_container.children = [secondary_button]
        state['secondary_prevention_visible'] = False

# Event handler for the 'Yes' button click
# Event handler for the 'Yes' button click
# Event handler for the 'Yes' button click
def on_yes_button_click(b, restore=False):
    state['yes_button'] = True
    state['no_button'] = False

    close_fh_branch()

    if not restore:
        state['current_state'] = 'yes_button'


    # Clear anything left over from Primary Prevention → No pathway
    age_input_container.children = []
    ckd_info_box_container.children = []
    ckd_yes_section_container.children = []
    diabetes_question_container.children = []
    diabetes_info_box_container.children = []
    diabetes_yes_section_container.children = []

    state['selected_age_option'] = None
    state['flowchart_created'] = False
    state['ckd_flowchart_created'] = False
    state['diabetes_question_visible'] = False
    state['diabetes_yes_section_visible'] = False
    state['oval_box_visible'] = False
    

    # Highlight 'Yes' button
    highlight_button([no_button, yes_button], yes_button)
    state['selected_button'] = 'yes_button'
    start_primary_to_secondary_glow()

    # Ensure the Primary Prevention branch stays visible
    if not state['primary_prevention_visible']:
        state['primary_prevention_visible'] = True
        update_primary_flowchart()

    # Make sure the Secondary Prevention branch is visible without selecting any option
    if not state['secondary_prevention_visible']:
        state['secondary_prevention_visible'] = True
        update_secondary_flowchart()  # Display the secondary prevention flowchart

    # Do not automatically select the "Yes" button in the Secondary Prevention branch
    # Simply make the Secondary Prevention branch visible



# Event handler for the CKD 'No' button click


<!-- ## 11. CKD and diabetes button handlers

Handles CKD and diabetes answers, including routing toward Type 1 and Type 2 diabetes branches. -->

In [12]:
def on_ckd_no_button_click(b, restore=False):
    state['ckd_no_button'] = True
    state['ckd_yes_button'] = False

    if not restore:
        state['current_state'] = 'ckd_no_button'

    highlight_button([ckd_no_button, ckd_yes_button, ckd_info_button], ckd_no_button)

    # Ensure the CKD yes section is hidden first
    clear_ckd_yes_section()
    state['ckd_yes_section_visible'] = False

    # Display diabetes question
    if not state['diabetes_question_visible']:
        display_diabetes_question()
    state['diabetes_question_visible'] = True

    # Save state
    state['ckd_state'] = 'ckd_no_button'



# Event handler for the CKD 'Yes' button click
def on_ckd_yes_button_click(b, restore=False):
    state['ckd_yes_button'] = True
    state['ckd_no_button'] = False

    if not restore:
        state['current_state'] = 'ckd_yes_button'

    highlight_button([ckd_no_button, ckd_yes_button, ckd_info_button], ckd_yes_button)

    # Clear diabetes question if it was previously displayed
    clear_diabetes_question()
    state['diabetes_question_visible'] = False

    # Display CKD yes treatment section
    display_ckd_yes_section()
    state['ckd_yes_section_visible'] = True

    # Save state
    state['ckd_state'] = 'ckd_yes_button'



# Event handler for the diabetes 'No' button click
def on_diabetes_no_button_click(b, restore=False):
    state['diabetes_no_button'] = True
    state['diabetes_yes_button'] = False
    state['diabetes_yes_section_visible'] = False
    state['type1_selected'] = False
    state['type2_selected'] = False
    state['type1_options_visible'] = False
    state['type1_any_yes'] = False
    state['type1_state'] = None

    if not restore:
        state['current_state'] = 'diabetes_no_button'

    highlight_button(
        [diabetes_no_button, diabetes_yes_button, diabetes_info_button],
        diabetes_no_button
    )

    # This removes Type 1 / Type 2 diabetes buttons
    clear_diabetes_yes_section()

    # Clear any Type 1 diabetes outputs/advice boxes
    clear_additional_section()

    # Then go straight to QRISK3
    display_qrisk3_section(show_diabetes_type_buttons=False)

    state['diabetes_no_state'] = {
        'qrisk3_section_visible': True
    }


# Event handler for the diabetes 'Yes' button click
def on_diabetes_yes_button_click(b, restore=False):
    state['diabetes_yes_button'] = True
    state['diabetes_no_button'] = False

    if not restore:
        state['current_state'] = 'diabetes_yes_button'

    highlight_button([diabetes_no_button, diabetes_yes_button, diabetes_info_button], diabetes_yes_button)

    # Display the diabetes 'Yes' section only if not already visible
    if not state['diabetes_yes_section_visible']:
        display_diabetes_yes_section()
        state['diabetes_yes_section_visible'] = True

    # Restore Type 1 diabetes state if previously selected
    if state['type1_state']:
        restore_type1_state()

# Function to display options for Type 1 diabetes



def display_type1_no_criteria_advice():
    # Prevent duplicate no-criteria advice boxes
    if any(
        "type1-no-criteria-advice-box" in getattr(child, "_dom_classes", ())
        for child in diabetes_yes_section_container.children
    ):
        return

    no_criteria_box = widgets.HTML(
        value=f"""
        <div class="oval-box" style="
            background-color:#f2f2f2;
            font-size:15px;
            line-height:1.55;
            width:500px;
            box-sizing:border-box;
            margin:14px auto;
            text-align:center;
        ">
            <b>Type 1 diabetes: no automatic statin indication identified</b><br><br>
    
            This person does not meet the NICE criteria for routinely offering statin therapy
            based on age &gt;40 years, diabetes duration &gt;10 years, established nephropathy,
            or other cardiovascular risk factors.<br><br>
    
            Continue lifestyle advice and optimise modifiable cardiovascular risk factors,
            including smoking, diet, weight management, physical activity, blood pressure
            control and glycaemic control.<br><br>
    
            NICE recommends considering statin therapy for all adults with Type 1 diabetes.
            Shared decision making should take into account the person's overall cardiovascular
            risk, preferences and clinical circumstances.<br><br>
    
            <a href="{CLINICAL_LINKS['NICE_NG238']}"
               target="_blank"
               style="color:#0066cc; font-weight:bold; text-decoration:underline;">
               NICE NG238 lipid modification guidance
            </a>
        </div>
        """
    )


    no_criteria_box.add_class("type1-no-criteria-advice-box")

    diabetes_yes_section_container.children += (no_criteria_box,)



def display_type1_options():
    type1_question = widgets.HTML(
        '<div class="option-title" style="text-align: center;"><u>Do they have the following?</u></div>'
    )

    options = [
        'Over 40 years of age',
        'Diabetes >10 years duration',
        'Established nephropathy',
        'Other risk factors for cardiovascular disease'
    ]

    radio_buttons = []

    for option in options:
        radio_button = widgets.RadioButtons(
            options=['No', 'Yes'],
            value='No',
            layout=widgets.Layout(width='auto')
        )

        line = widgets.HTML(
            value="<div style='border-bottom: 1px solid #ccc; width: 100%;'></div>"
        )

        option_label = widgets.HTML(
            value=f"""
            <div style="
                width:300px;
                text-align:left;
                font-weight:bold;
                font-size:16px;
            ">
                {option}
            </div>
            """
        )

        radio_buttons.append(
            widgets.VBox([
                widgets.HBox(
                    [option_label, radio_button],
                    layout=widgets.Layout(
                        justify_content='center',
                        align_items='center',
                        width='100%'
                    )
                ),
                line
            ])
        )

        radio_button.observe(on_radio_button_change, names='value')

    global type1_options_container
    type1_options_container = widgets.VBox(
        radio_buttons,
        layout=widgets.Layout(
            align_items='flex-start',
            width='auto'
        )
    )


    confirm_none_button = widgets.Button(
        description="Enter",
        tooltip="Lock in Type 1 diabetes selection",
        layout=widgets.Layout(width="150px", height="40px")
    )
    
    confirm_none_button.style.button_color = "#4CAF50"
    confirm_none_button.style.text_color = "white"
    confirm_none_button.add_class("green-pulse-button")
    


    
    def on_confirm_none_clicked(b):
        radio_buttons_widgets = [
            child.children[0].children[1]
            for child in type1_options_container.children
        ]

        any_yes = any(rb.value == 'Yes' for rb in radio_buttons_widgets)

        clear_additional_section()

        if any_yes:
            display_additional_section()
        else:
            display_type1_no_criteria_advice()

    confirm_none_button.on_click(on_confirm_none_clicked)

    global type1_oval_box
    type1_oval_box = widgets.VBox(
        [
            type1_question,
            type1_options_container,
            widgets.HTML('<div style="height:10px;"></div>'),
            widgets.HBox(
                [confirm_none_button],
                layout=widgets.Layout(
                    justify_content='center',
                    width='100%'
                )
            )

            
        ],
        layout=widgets.Layout(
            padding='20px',
            border='2px solid black',
            border_radius='25px',
            background_color='#f0f0f0',
            font_size='18px',
            margin='10px',
            text_align='left',
            width='fit-content',
            align_self='center'
        )
    )

    diabetes_yes_section_container.children = [
        widgets.HBox(
            [type1_button, type2_button],
            layout=widgets.Layout(
                justify_content='center',
                width='100%'
            )
        ),
        type1_oval_box
    ]

    state['type1_options_visible'] = True
    


# Function to handle the radio button change event
def on_radio_button_change(change):
    radio_buttons_widgets = [
        child.children[0].children[1]
        for child in type1_options_container.children
    ]

    any_yes = any(rb.value == 'Yes' for rb in radio_buttons_widgets)

    state['type1_any_yes'] = any_yes

    clear_additional_section()

    if any_yes:
        display_additional_section()

# Function to check the state of the radio buttons
# Function to check the state of the radio buttons
def check_radio_buttons():
    # Extract the RadioButtons widgets from the VBox children
    radio_buttons_widgets = [child.children[0].children[1] for child in type1_options_container.children]

    # Check if all RadioButtons are set to 'No'
    if all(rb.value == 'No' for rb in radio_buttons_widgets):
        state['type1_any_yes'] = False
        clear_additional_section()



# Function to display the additional section below the radio button oval box
def display_additional_section():
    TYPE1_OUTPUT_WIDTH = '500px'
    # Prevent duplicate Type 1 advice sections being added
    if any(
        "type1-advice-box" in getattr(child, "_dom_classes", ())
        for child in diabetes_yes_section_container.children
    ):
        return


    modifiable_examples = widgets.HTML(
        value="""
        <div style="
            text-align:left;
            font-size:14px;
            line-height:1.6;
            margin-top:8px;
            padding:10px 14px;
            border:1px solid #cccccc;
            border-radius:10px;
            background-color:#f8f8f8;
        ">
            <strong>Examples:</strong>
            <ul style="margin-top:6px; margin-bottom:0;">
                <li>Smoking</li>
                <li>Diet</li>
                <li>Obesity</li>
                <li>Alcohol intake</li>
                <li>Physical activity</li>
                <li>Blood pressure</li>
                <li>Glycaemic control / HbA1c</li>
            </ul>
        </div>
        """
    )

    additional_risk_examples = widgets.HTML(
        value="""
        <div style="
            text-align:left;
            font-size:14px;
            line-height:1.6;
            margin-top:8px;
            padding:10px 14px;
            border:1px solid #cccccc;
            border-radius:10px;
            background-color:#f8f8f8;
        ">
            <strong>Examples:</strong>
            <ul style="margin-top:6px; margin-bottom:0;">
                <li>Treated for HIV</li>
                <li>Severe mental illness</li>
                <li>Taking medicines that cause dyslipidaemia</li>
                <li>Systemic inflammatory disorder</li>
                <li>Impaired fasting glycaemia</li>
            </ul>
        </div>
        """
    )

    modifiable_examples_box = widgets.VBox([])
    additional_risk_examples_box = widgets.VBox([])

    
    modifiable_button = widgets.Button(
        description="Show examples",
        layout=widgets.Layout(width="130px", height="28px")
    )
    
    additional_risk_button = widgets.Button(
        description="Show examples",
        layout=widgets.Layout(width="130px", height="28px")
    )
    
    def toggle_modifiable_examples(b):
        if len(modifiable_examples_box.children) == 0:
            modifiable_examples_box.children = [modifiable_examples]
            modifiable_button.description = "Hide examples"
        else:
            modifiable_examples_box.children = []
            modifiable_button.description = "Show examples"

    def toggle_additional_risk_examples(b):
        if len(additional_risk_examples_box.children) == 0:
            additional_risk_examples_box.children = [additional_risk_examples]
            additional_risk_button.description = "Hide examples"
        else:
            additional_risk_examples_box.children = []
            additional_risk_button.description = "Show examples"

    modifiable_button.on_click(toggle_modifiable_examples)
    additional_risk_button.on_click(toggle_additional_risk_examples)

    
    type1_advice_box = widgets.VBox(
        [
            widgets.HTML(
                value="""
                <div style="
                    font-size:16px;
                    font-weight:bold;
                    text-align:center;
                    margin-bottom:14px;
                ">
                    <u>Type 1 diabetes: Primary prevention statin treatment advice</u>
                </div>
                """
            ),
    
            widgets.HBox(
                [
                    widgets.HTML(
                        value="""
                        <div style="font-size:15px; line-height:1.65;">
                            <strong>1. Discuss benefits of lifestyle changes and address all modifiable risk factors.</strong>
                        </div>
                        """,
                        layout=widgets.Layout(width="365px")
                    ),
                    modifiable_button
                ],
                layout=widgets.Layout(
                    justify_content="flex-start",
                    align_items="center",
                    width="100%",
                    gap="6px"
                )
            ),
            modifiable_examples_box,
    
            widgets.HBox(
                [
                    widgets.HTML(
                        value="""
                        <div style="font-size:15px; line-height:1.65; margin-top:10px;">
                            <strong>2. Consider additional risk factors if present.</strong>
                        </div>
                        """,
                        layout=widgets.Layout(width="330px")
                    ),
                    additional_risk_button
                ],
                layout=widgets.Layout(
                    justify_content="flex-start",
                    align_items="center",
                    width="100%",
                    gap="6px"
                )
            ),
            additional_risk_examples_box,
    
            widgets.HTML(
                value="""
                <div style="font-size:15px; line-height:1.65; margin-top:10px;">
                    <strong>3. Offer Atorvastatin 20mg once daily.</strong>
                </div>
                """
            ),
        ],
        layout=widgets.Layout(
            width=TYPE1_OUTPUT_WIDTH,
            box_sizing="border-box",
            margin="12px auto",
            padding="16px 18px",
            border="2px solid black",
            border_radius="25px",
            background_color="#ffffff"
        )
    )
    
    type1_advice_box.add_class("type1-advice-box")
    
    
    
    
    





    
    non_hdl_label = widgets.HTML(
        value="""
        <div style="
            font-size:15px;
            font-weight:500;
            white-space:nowrap;
            margin-right:8px;
        ">
            Baseline Non-HDL (mmol/L):
        </div>
        """
    )
    
    non_hdl_input = widgets.Text(
        value='',
        placeholder='e.g. 4.8',
        layout=widgets.Layout(width='70px')
    )




    
    target_output = widgets.HTML(
        value='',
        layout=widgets.Layout(width='100%')
    )

    non_hdl_warning = widgets.HTML(
        value='',
        layout=widgets.Layout(width='100%')
    )

    non_hdl_enter_button = widgets.Button(
        description="Enter",
        layout=widgets.Layout(width='85px', height='30px')
    )

    def calculate_type1_target(b):
        raw_value = non_hdl_input.value.strip()

        target_output.value = ''
        non_hdl_warning.value = ''

        try:
            baseline = float(raw_value)
        except ValueError:
            non_hdl_warning.value = """
            <div style="
                color:#b00020;
                font-weight:bold;
                font-size:15px;
                text-align:center;
                margin-top:8px;
            ">
                Invalid Non-HDL. Please enter a valid number.
            </div>
            """
            return

        if baseline < 0:
            non_hdl_warning.value = """
            <div style="
                color:#b00020;
                font-weight:bold;
                font-size:15px;
                text-align:center;
                margin-top:8px;
            ">
                Invalid Non-HDL. Please enter a valid number.
            </div>
            """
            return

        target = baseline * 0.6

        target_output.value = f"""
        <div style="
            font-size:18px;
            font-weight:bold;
            text-align:center;
            margin-top:10px;
        ">
            <span style="text-decoration:underline;">
                Target Non-HDL:
            </span>
            {target:.2f} mmol/L
        </div>
        """

    non_hdl_enter_button.on_click(calculate_type1_target)
    
    target_non_hdl_input_row = widgets.HBox(
        [non_hdl_label, non_hdl_input, non_hdl_enter_button],
        layout=widgets.Layout(
            justify_content='center',
            align_items='center',
            width='100%',
            margin='6px 0 2px 0',
            overflow='visible'
        )
    )

    target_non_hdl_box = widgets.VBox([
        widgets.HTML("""
        <div class="target-nonhdl-title">
            For Target Non-HDL cholesterol, please enter baseline 
            pre-treatment Non-HDL here:
        </div>
        """),
        target_non_hdl_input_row,
        non_hdl_warning,
        target_output
    ],
    layout=widgets.Layout(
        align_items='center',
        width=QRISK3_SECTION_WIDTH,
        margin='12px auto',
        overflow='visible'
    ))

    target_non_hdl_box.add_class("target-nonhdl-box")
    
    follow_up_box = widgets.HTML(
        """
        <div class="followup-green-box">
            Measure lipid profile after 3 months and aim for the above Non-HDL cholesterol target.
        </div>
        """,
        layout=widgets.Layout(
            width=TYPE1_OUTPUT_WIDTH,
            margin='12px auto'
        )
    )

    
    diabetes_yes_section_container.children += (
        type1_advice_box,
        target_non_hdl_box,
        follow_up_box
    )

    
    def on_lifestyle_expand(b):
        if len(lifestyle_box.children) == 2:
            lifestyle_box.children = list(lifestyle_box.children) + [widgets.HTML('<div style="text-align: left;">Smoking, diet, obesity, alcohol intake, physical activity, blood pressure and HbA1c</div>')]
            lifestyle_box.children[0].layout.height = 'auto'
        else:
            lifestyle_box.children = lifestyle_box.children[:2]
            lifestyle_box.children[0].layout.height = 'auto'

    def on_additional_risk_expand():
        if len(additional_risk_box.children) == 2:
            additional_risk_box.children = list(additional_risk_box.children) + [widgets.HTML('''
            <div style="text-align: left;">
                Treated for HIV, severe mental illness, taking medicines that cause dyslipidaemia,<br>

                systemic inflammatory disorder, impaired fasting glycaemia<br>

            </div>
            ''')]
            additional_risk_box.children[0].layout.height = 'auto'
        else:
            additional_risk_box.children = additional_risk_box.children[:2]
            additional_risk_box.children[0].layout.height = 'auto'

    lifestyle_box.children[1].on_click(on_lifestyle_expand)
    additional_risk_box.children[1].on_click(on_additional_risk_expand)


def clear_additional_section():
    children_to_keep = []

    for child in diabetes_yes_section_container.children:

        # New expandable Type 1 advice VBox
        is_new_type1_advice_box = (
            "type1-advice-box" in getattr(child, "_dom_classes", ())
        )

        # Old Type 1 advice HTML box, kept here for safety
        is_old_type1_advice_box = (
            isinstance(child, widgets.HTML)
            and "Type 1 diabetes:" in child.value
            and "Primary prevention statin" in child.value
        )

        is_target_non_hdl_box = (
            isinstance(child, widgets.VBox)
            and "target-nonhdl-box" in getattr(child, "_dom_classes", ())
        )

        is_follow_up_box = (
            isinstance(child, widgets.HTML)
            and "Measure lipid profile after 3 months" in child.value
        )

        is_type1_no_criteria_advice_box = (
            "type1-no-criteria-advice-box" in getattr(child, "_dom_classes", ())
        )
        

        if (
            not is_new_type1_advice_box
            and not is_old_type1_advice_box
            and not is_target_non_hdl_box
            and not is_follow_up_box
            and not is_type1_no_criteria_advice_box
        ):

            
            children_to_keep.append(child)

    diabetes_yes_section_container.children = children_to_keep



def restore_type1_options():
    if state['type1_selected']:
        display_type1_options()


    
# Event handler for the 'Type 1 diabetes?' button click
def on_type1_button_click(b):
    state['type1_selected'] = True
    state['type2_selected'] = False
    highlight_button([type1_button, type2_button], type1_button)
    if not state['type1_options_visible']:
        display_type1_options()
    else:
        restore_type1_options()
    # Save state for Type 1 options
    state['type1_state'] = {
        'type1_selected': state['type1_selected'],
        'type2_selected': state['type2_selected'],
        'type1_options_visible': state['type1_options_visible'],
        'type1_any_yes': state['type1_any_yes']
    }



def on_type2_button_click(b):
    state['type1_selected'] = False
    state['type2_selected'] = True
    state['type1_options_visible'] = False
    state['type1_any_yes'] = False

    highlight_button([type1_button, type2_button], type2_button)

    # Clear Type 1 outputs before showing the Type 2 QRISK3 section
    clear_additional_section()

    # Reuse the existing QRISK3 section
    display_qrisk3_section()


# Function to clear type 1 options
def clear_type1_options():
    if type1_options_container:
        type1_options_container.children = []

# Function to clear the Type 2 box
def clear_type2_box():
    if len(diabetes_yes_section_container.children) > 2:
        diabetes_yes_section_container.children = diabetes_yes_section_container.children[:3]  # Keep the vertical line and the buttons

# Function to clear diabetes question
def clear_diabetes_question():
    if state['diabetes_question_visible']:
        primary_flowchart_container.children = [child for child in primary_flowchart_container.children if child != diabetes_question_container]
        state['diabetes_question_visible'] = False

# Function to clear diabetes yes section
def clear_diabetes_yes_section():
    diabetes_yes_section_container.children = []
    state['diabetes_yes_section_visible'] = False

<!-- ## 12. FH button, Secondary Prevention toggle, and reset helpers

Connects the FH referral button, toggles Secondary Prevention, and resets competing branches. -->

In [13]:
cholesterol_input_container = widgets.VBox([], layout=widgets.Layout(align_items='center'))

def close_fh_branch():
    fh_container.children = [hyperlipidaemia_button]
    state["fh_line_visible"] = False
    
def on_hyperlipidaemia_button_click(b):
    state['primary_prevention'] = False
    state['secondary_prevention'] = False

    if state.get("fh_line_visible", False):
        fh_container.children = [hyperlipidaemia_button]
        state["fh_line_visible"] = False
    else:
        fh_line_and_input = build_fh_input_section()
        fh_container.children = [hyperlipidaemia_button, fh_line_and_input]
        state["fh_line_visible"] = True

hyperlipidaemia_button.on_click(on_hyperlipidaemia_button_click)

# For demonstration, we display the FH container centered on the page.


# Do not recreate button_container here.
# The main scrollable three-column layout is created once in Section 7.


# Event handler for the 'Secondary Prevention' button click
def reset_flowchart():
    # Clear all containers and reset visibility flags
    primary_flowchart_container.children = [primary_button]
    age_input_container.children = []
    ckd_info_box_container.children = []
    diabetes_question_container.children = []
    diabetes_yes_section_container.children = []
    secondary_flowchart_container.children = [secondary_button]
    acs_flowchart_container.children = []

    drug_interaction_flowchart_container.children = [
        child for child in drug_interaction_flowchart_container.children
        if not (
            isinstance(child, widgets.VBox) and any(
                isinstance(grandchild, widgets.HTML)
                and f'Is Non-HDL cholesterol less than {NON_HDL_TREATMENT_THRESHOLDS["SECONDARY_TARGET"]} mmol/L?' in grandchild.value
                for grandchild in child.children
            )
        )
    ]

    # Reset state variables
    state.update({
        'primary_prevention_visible': False,
        'no_button': False,
        'yes_button': False,
        'flowchart_created': False,
        'ckd_flowchart_created': False,
        'diabetes_question_visible': False,
        'diabetes_yes_section_visible': False,
        'secondary_prevention_visible': False,
        'acs_flowchart_created': False,
        'drug_interaction_flowchart_created': False,
        'acs_oval_box_visible': False,
        'modifiable_risk_factors_oval_box_visible': False,
        'drug_interaction_oval_visible': False,

        'non_hdl_flowchart_created': False,   # Track Non-HDL flowchart state
        'non_hdl_yes_button': False,         # Track Non-HDL "Yes" button state
        'non_hdl_no_button': False,          # Track Non-HDL "No" button state
        'non_hdl_info_box_visible': False,
    })

    # Restore initial flowchart structure
    update_primary_flowchart()
    update_secondary_flowchart()
    
def clear_secondary_flowchart():
    # Clear any children in the secondary flowchart container before adding new items
    secondary_flowchart_container.children = [secondary_button]
    state['acs_flowchart_created'] = False


<!-- ## 13. Acute coronary syndrome branch

Builds and updates the ACS question and related statin/risk-factor recommendation boxes. -->

In [14]:
def create_acs_flowchart():
    # Clear the flowchart if already created to avoid duplication
    clear_acs_flowchart()

    # Add the ACS question if it hasn't been created yet
    if not state['acs_flowchart_created']:
        acs_question_html = widgets.VBox([
            widgets.HTML('<div style="font-size: 14px; text-align: center;">Is the person being treated for acute coronary syndrome currently?</div>'),
            widgets.HBox([
                acs_yes_button,
                widgets.HTML('<div class="horizontal-line"></div>'),
                acs_info_button,
                widgets.HTML('<div class="horizontal-line"></div>'),
                acs_no_button
            ], layout=widgets.Layout(justify_content='center', align_items='center', width='350px')),
            acs_info_box_container
        ], layout=widgets.Layout(align_items='center'))

        acs_flowchart_container.children = [widgets.HTML('<div class="vertical-line"></div>'), acs_question_html]
        state['acs_flowchart_created'] = True  # Set flag to prevent duplication



# Ensure that acs_oval_box_visible is properly tracked, and check the restore flag to manage state


state['acs_oval_visible'] = False  # Initialize state variable

# Function to show ACS oval box
def show_acs_oval_box():
    if not state.get('acs_oval_box_visible', False):
        # Create the oval box
        acs_oval_box = widgets.HTML('''

            <div class="oval-box" style="
                width: 400px;
                box-sizing: border-box;
                padding: 8px 20px;
                border: 2px solid black;
                border-radius: 25px;
                background-color: #FFFFFF;
                font-size: 14px;
                margin: 10px auto;
                text-align: center;">
                <strong>1. Do not delay statin treatment</strong><br>
                <strong>2. Take a lipid profile on admission</strong> (within 24 hours)
            </div>
        ''')

        # Inject the oval at the correct position (e.g., after Step 4)
        position = 2  # Adjust this index based on where it should appear
        acs_flowchart_container.children = (
            list(acs_flowchart_container.children[:position])
            + [acs_oval_box]
            + list(acs_flowchart_container.children[position:])
        )
        state['acs_oval_box_visible'] = True

def hide_acs_oval_box():
    if state.get('acs_oval_box_visible', False):
        # Remove the oval box if visible
        acs_flowchart_container.children = [
            child for child in acs_flowchart_container.children
            if not (isinstance(child, widgets.HTML) and "Do not delay statin treatment" in child.value)
        ]
        state['acs_oval_box_visible'] = False


# [V2 cleanup] Removed first overwritten update_acs_oval_visibility definition; later definition is the active one.
# state['acs_oval_box_retained'] = False

def on_acs_yes_button_click(b):
    state['acs_yes_button'] = True
    state['acs_no_button'] = False
    highlight_button([acs_yes_button, acs_no_button, acs_info_button], acs_yes_button)

    # Hide the "Offer statin therapy" oval
    hide_statin_oval_box()
    hide_statin_oval_box()

    # Update the visibility of other ACS-related ovals
    update_acs_oval_visibility()

    
def show_statin_oval_box():
    if not state.get('statin_oval_box_visible', False):
        # Create the Step 5 oval box
        statin_oval_box = widgets.HTML('''
            <div class="oval-box" style="
                padding: 8px 10px;
                border: 2px solid black;
                border-radius: 25px;
                background-color: #FFFFFF;
                font-size: 13px;
                margin: 10px auto;
                width: 250px;
                text-align: center;">
                <strong>Offer statin therapy</strong>
            </div>
        ''')

        # Inject the oval above "Identify and address modifiable risk factors"
        position = 2  # Adjust this index to place above modifiable risk factors
        acs_flowchart_container.children = (
            list(acs_flowchart_container.children[:position])
            + [statin_oval_box]
            + list(acs_flowchart_container.children[position:])
        )
        state['statin_oval_box_visible'] = True


def hide_statin_oval_box():
    if state.get('statin_oval_box_visible', False):
        acs_flowchart_container.children = [
            child for child in acs_flowchart_container.children
            if not (isinstance(child, widgets.HTML) and "Offer statin therapy" in child.value)
        ]
        state['statin_oval_box_visible'] = False


def update_acs_oval_visibility():
    # Check if ACS "Yes" is selected
    if state.get('acs_yes_button', False) and state.get('secondary_yes_button', False):
        if not state.get('acs_oval_box_visible', False):
            show_acs_oval_box()
        if not state.get('statin_oval_box_visible', False):  # Add Step 5 oval
            show_statin_oval_box()
        hide_statin_oval_box()
        show_modifiable_risk_factors_oval_box()
        create_drug_interaction_flowchart()
    # Check if ACS "No" is selected
    elif state.get('acs_no_button', False) and state.get('secondary_yes_button', False):
        hide_acs_oval_box()
        show_statin_oval_box()  # Hide Step 5 oval
        show_modifiable_risk_factors_oval_box()
        create_drug_interaction_flowchart()
    else:
        hide_acs_oval_box()
        hide_statin_oval_box()  # Hide Step 5 oval
        hide_modifiable_risk_factors_oval_box()
        drug_interaction_flowchart_container.children = []  # Clear drug interaction container if no valid selection



def on_acs_no_button_click(b):
    state['acs_yes_button'] = False
    state['acs_no_button'] = True
    highlight_button([acs_yes_button, acs_no_button, acs_info_button], acs_no_button)

    # Ensure neither ACS-specific nor "Offer statin therapy" ovals are displayed
    hide_acs_oval_box()
    hide_statin_oval_box()

    # Update any additional logic related to ACS "No" state
    update_acs_oval_visibility()



<!-- ## 14. Drug-interaction branch and secondary-state reset helpers

Handles Yes/No answers to the drug-interaction question and restores dependent secondary-prevention branches. -->

In [15]:
# Show the drug interaction branch based on ACS state
def on_drug_interaction_yes_button_click(b, restore=False):
    state['drug_interaction_yes_button'] = True
    state['drug_interaction_no_button'] = False
    highlight_button(
        [drug_interaction_yes_button, drug_interaction_no_button, drug_interaction_info_button],
        drug_interaction_yes_button
    )

    existing_children = list(drug_interaction_flowchart_container.children)

    # Remove old ovals
    def is_old_oval(child):
        return (
            isinstance(child, widgets.HTML)
            and (
                "Consider using a lower dose of Atorvastatin" in child.value
                or "Prescribe Atorvastatin 80mg daily" in child.value
                or "Measure non-fasting lipid profile after 3 months" in child.value
            )
        )
    cleaned_children = [c for c in existing_children if not is_old_oval(c)]

    # Find vertical line to insert above
    insert_index = None
    for i, child in enumerate(cleaned_children):
        if isinstance(child, widgets.HTML) and '<div class="vertical-line"></div>' in child.value:
            insert_index = i
            break
    if insert_index is None:
        insert_index = len(cleaned_children)

    #
    # IMPORTANT: match the ACS inline style EXACTLY:
    #
    lower_dose_oval = widgets.HTML('''
        <div class="oval-box" style="
            padding: 20px 10px;
            border: 2px solid black;
            border-radius: 25px;
            background-color: #FFFFFF;
            font-size: 13px;
            margin: 10px auto;
            width: 300px;
            text-align: center;">
            <strong>Consider using a lower dose of Atorvastatin</strong>
        </div>
    ''')
    measure_oval = widgets.HTML('''
        <div class="oval-box" style="
            padding: 20px 20px;
            border: 2px solid black;
            border-radius: 25px;
            background-color: #FFFFFF;
            font-size: 13px;
            margin: 10px auto;
            width: 320px;
            text-align: center;">
            <strong>Measure non-fasting lipid profile after 3 months</strong>
        </div>
    ''')

    # Insert them


    cleaned_children[insert_index:insert_index] = [lower_dose_oval, measure_oval]
    drug_interaction_flowchart_container.children = tuple(cleaned_children)

    # Trigger Non-HDL if needed
    if not state['non_hdl_flowchart_created']:
        trigger_non_hdl_flowchart()


def on_drug_interaction_no_button_click(b, restore=False):
    state['drug_interaction_yes_button'] = False
    state['drug_interaction_no_button'] = True
    highlight_button(
        [drug_interaction_yes_button, drug_interaction_no_button, drug_interaction_info_button],
        drug_interaction_no_button
    )

    existing_children = list(drug_interaction_flowchart_container.children)

    def is_old_oval(child):
        return (
            isinstance(child, widgets.HTML)
            and (
                "Consider using a lower dose of Atorvastatin" in child.value
                or "Prescribe Atorvastatin 80mg daily" in child.value
                or "Measure non-fasting lipid profile after 3 months" in child.value
            )
        )
    cleaned_children = [c for c in existing_children if not is_old_oval(c)]

    insert_index = None
    for i, child in enumerate(cleaned_children):
        if isinstance(child, widgets.HTML) and '<div class="vertical-line"></div>' in child.value:
            insert_index = i
            break
    if insert_index is None:
        insert_index = len(cleaned_children)

    #
    # EXACT same inline style as the ACS ovals:
    #
    prescribe_oval = widgets.HTML('''
        <div class="oval-box" style="
            padding: 20px 10px;
            border: 2px solid black;
            border-radius: 25px;
            background-color: #FFFFFF;
            font-size: 13px;
            margin: 10px auto;
            width: 270px;
            text-align: center;">
            <strong>Prescribe Atorvastatin 80mg daily</strong>
        </div>
    ''')
    measure_oval = widgets.HTML('''
        <div class="oval-box" style="
            padding: 20px 10px;
            border: 2px solid black;
            border-radius: 25px;
            background-color: #FFFFFF;
            font-size: 13px;
            margin: 10px auto;
            width: 300px;
            text-align: center;">
            <strong>Measure non-fasting lipid profile after 3 months</strong>
        </div>
    ''')

    cleaned_children[insert_index:insert_index] = [prescribe_oval, measure_oval]
    drug_interaction_flowchart_container.children = tuple(cleaned_children)
    

    if not state['non_hdl_flowchart_created']:
        trigger_non_hdl_flowchart()



def trigger_non_hdl_flowchart():
    """Trigger the Non-HDL flowchart based on the Drug Interaction state."""
    # Ensure Drug Interaction flowchart was completed
    if state.get('drug_interaction_yes_button', False) or state.get('drug_interaction_no_button', False):
        if not state.get('non_hdl_flowchart_created', False):
            #print("Creating Non-HDL flowchart...")
            create_non_hdl_flowchart()
        else:
           # print("Non-HDL flowchart already exists. Restoring visibility...")
            restore_non_hdl_flowchart_visibility()  # Ensure it is visible if toggled off
    else:
        print("")
     #   print("Drug Interaction flowchart not completed. Cannot activate Non-HDL flowchart.")


def restore_non_hdl_flowchart_visibility():
    """Restore the Non-HDL flowchart visibility if it was toggled off."""

    NON_HDL_TARGET_TEXT = (
        f'Is Non-HDL cholesterol less than '
        f'{NON_HDL_TREATMENT_THRESHOLDS["SECONDARY_TARGET"]} mmol/L?'
    )

    if not any(
        isinstance(child, widgets.VBox) and any(
            isinstance(grandchild, widgets.HTML)
            and NON_HDL_TARGET_TEXT in grandchild.value
            for grandchild in child.children
        )
        for child in drug_interaction_flowchart_container.children
    ):
        # Re-add the Non-HDL flowchart if missing
        vertical_line_html = widgets.HTML('<div class="vertical-line"></div>')
        non_hdl_question_html = widgets.VBox([
            
            widgets.HTML(
                f'<div style="font-size: 14px; text-align: center;">'
                f'Is Non-HDL cholesterol less than '
                f'{NON_HDL_TREATMENT_THRESHOLDS["SECONDARY_TARGET"]} mmol/L?'
                f'</div>'
            ),
            
            widgets.HBox([
                non_hdl_yes_button,
                widgets.HTML('<div class="horizontal-line"></div>'),
                non_hdl_info_button,
                widgets.HTML('<div class="horizontal-line"></div>'),
                non_hdl_no_button
            ], layout=widgets.Layout(justify_content='center', align_items='center', width='350px')),
            non_hdl_info_box_container
        ], layout=widgets.Layout(align_items='center'))

        # Add the Non-HDL question to the Drug Interaction flowchart container
        drug_interaction_flowchart_container.children += (vertical_line_html, non_hdl_question_html, non_hdl_flowchart_container)

       # print("Non-HDL flowchart visibility restored.")



def refresh_container(container):
    children = list(container.children)
    container.children = []
    container.children = children
   # print("Container refreshed.")




def toggle_drug_interaction_info_box(b):
    state['drug_interaction_info_box_visible'] = not state['drug_interaction_info_box_visible']
    if state['drug_interaction_info_box_visible']:
        drug_interaction_info_box = widgets.HTML('''
        <div class="info-box">
            Information about potential drug interactions or adverse effects to high-intensity statin.
        </div>
        ''')
        drug_interaction_info_box_container.children = [drug_interaction_info_box]
    else:
        drug_interaction_info_box_container.children = []





def on_cvd_toggle():
    # Existing toggle logic for CVD here
    update_acs_oval_visibility()




# Function to toggle the ACS info box
def toggle_acs_info_box(b):
    state['acs_info_box_visible'] = not state['acs_info_box_visible']
    if state['acs_info_box_visible']:
        acs_info_box = widgets.HTML('''
        <div class="info-box">
            <strong>Acute Coronary Syndrome (ACS)</strong><br>
            ACS includes unstable angina, ST-elevation myocardial infarction (STEMI), and non-ST-elevation myocardial infarction (NSTEMI).
        </div>
        ''')
        acs_info_box_container.children = [acs_info_box]
    else:
        acs_info_box_container.children = []


def reset_acs_state():
    state['acs_yes_button'] = False
    state['acs_no_button'] = False
    state['acs_info_box_visible'] = False
    state['acs_oval_box_visible'] = False
    clear_acs_flowchart()  # Ensure ACS flowchart is cleared

def reset_secondary_prevention_state():
    state['secondary_yes_button'] = False
    state['secondary_no_button'] = False
    reset_acs_state()  # Ensure that ACS states are reset when switching Secondary Prevention options


In [16]:
# <!-- ## 15. Secondary Prevention CVD routing and base UI

# Handles secon|dary-prevention Yes/No answers and creates modifiable-risk-factor information. -->

In [17]:

def on_secondary_button_click(b):
    if state['secondary_prevention_visible']:
        # Hide the flowchart
        secondary_flowchart_container.children = [secondary_button]
        state['secondary_prevention_visible'] = False
    else:
        # Show the flowchart
        state['current_state'] = 'secondary_prevention'
        state['secondary_prevention_visible'] = True
        state['secondary_yes_button'] = False
        state['secondary_no_button'] = False
        update_secondary_flowchart()

def on_secondary_yes_button_click(b, restore=False):
    state['secondary_yes_button'] = True
    state['secondary_no_button'] = False

    if not restore:
        state['current_state'] = 'secondary_yes_button'

    # Stop the reverse glow from Secondary No → Primary Prevention
    stop_secondary_to_primary_glow()

    # Stop the opposite Primary → Secondary glow too, just in case
    stop_primary_to_secondary_glow()

    # Highlight the 'Yes' button for CVD
    highlight_button([secondary_yes_button, secondary_no_button], secondary_yes_button)

    # Hide the Primary Prevention branch when CVD is "Yes"
    if state['primary_prevention_visible']:
        primary_flowchart_container.children = [primary_button]
        state['primary_prevention_visible'] = False

    # Display the ACS branch
    if not state['acs_flowchart_created']:
        create_acs_flowchart()

    # Update ACS oval visibility
    update_acs_oval_visibility()

    restore_non_hdl()

def on_secondary_no_button_click(b, restore=False):
    # Update state for Secondary Prevention "No"
    state['secondary_no_button'] = True
    state['secondary_yes_button'] = False

    close_fh_branch()
    
    if not restore:
        state['current_state'] = 'secondary_no_button'

    # Highlight the Secondary Prevention "No" button
    highlight_button([secondary_yes_button, secondary_no_button], secondary_no_button)

    # Stop the opposite direction glow first
    stop_primary_to_secondary_glow()

    # Open/show the Primary Prevention branch first.
    # This may call stop_secondary_to_primary_glow(), so do it before starting the reverse glow.
    if not state['primary_prevention_visible']:
        on_primary_button_click(None)

    # Now start reverse glow: Secondary No → Primary Prevention
    start_secondary_to_primary_glow()

    # Clear Secondary Prevention downstream flowcharts
    clear_acs_flowchart()
    clear_non_hdl_flowchart()

    drug_interaction_flowchart_container.children = []
    state['drug_interaction_flowchart_created'] = False

    state['non_hdl_activated'] = False



def clear_acs_flowchart():
    acs_flowchart_container.children = []  # Clear the ACS flowchart container
    state['acs_flowchart_created'] = False  # Reset the flag so it can be recreated
    hide_acs_oval_box()
    hide_modifiable_risk_factors_oval_box()




def on_secondary_no_button_click(b, restore=False):
    # Update state for Secondary Prevention "No"
    state['secondary_no_button'] = True
    state['secondary_yes_button'] = False

    close_fh_branch()
    # Track current state unless restoring
    if not restore:
        state['current_state'] = 'secondary_no_button'

    # Highlight the Secondary Prevention "No" button
    highlight_button([secondary_yes_button, secondary_no_button], secondary_no_button)

    # Start reverse glow: Secondary No → Primary Prevention
    start_secondary_to_primary_glow()

    # Stop the opposite glow direction if it was active
    stop_primary_to_secondary_glow()

    # Show the Primary Prevention branch when Secondary CVD is "No"
    if not state['primary_prevention_visible']:
        on_primary_button_click(None)

    # Clear Secondary Prevention downstream flowcharts
    clear_acs_flowchart()
    clear_non_hdl_flowchart()

    drug_interaction_flowchart_container.children = []
    state['drug_interaction_flowchart_created'] = False

    state['non_hdl_activated'] = False


# Event handler for the 'Secondary Prevention' Info button click
def on_secondary_info_button_click(b):
    state['secondary_info_visible'] = not state.get('secondary_info_visible', False)

    if state['secondary_info_visible']:
        secondary_info_box_container.children = [
            widgets.HTML("""
            <div class="info-box">
                <b>Definition of cardiovascular disease (CVD):</b><br><br>

                <b>a. Coronary Heart Disease</b>, including Angina, Acute Coronary Syndrome,
                Myocardial Infarction, unstable angina or Coronary Revascularisation<br>

                <b>b. Stroke or transient ischaemic attack (TIA)</b><br>

                <b>c. Symptomatic peripheral arterial disease</b>
            </div>
            """)
        ]
    else:
        secondary_info_box_container.children = []

def update_secondary_flowchart():
    # Create the vertical line and question
    line_html = widgets.HTML('<div class="vertical-line"></div>')

    question_html = widgets.VBox([
        widgets.HTML('<div style="font-size: 14px; text-align: center;">Does this person have cardiovascular disease (CVD)?</div>'),
        widgets.HBox([
            secondary_yes_button,
            widgets.HTML('<div class="horizontal-line" style="flex-grow: 1;"></div>'),
            secondary_info_button,
            widgets.HTML('<div class="horizontal-line" style="flex-grow: 1;"></div>'),
            secondary_no_button
        ], layout=widgets.Layout(justify_content='center', align_items='center', width='350px')),
        secondary_info_box_container
    ], layout=widgets.Layout(align_items='center'))
    

    # Update the container to include the line, question, and buttons
    secondary_flowchart_container.children = [secondary_button, line_html, question_html, acs_flowchart_container]




# Function to show the new oval box with modifiable risk factors
def show_modifiable_risk_factors_oval_box():
    if not state.get('modifiable_risk_factors_oval_box_visible', False):
        modifiable_risk_factors_oval_box = widgets.HTML('''
            <div class="oval-box" style="
                padding: 8px 10px;
                border: 2px solid black;
                border-radius: 25px;
                background-color: #FFFFFF;
                font-size: 13px;
                width: 380px;
                margin: 10px auto;
                text-align: center;">

                <strong>Identify and address on modifiable risk factors:</strong><br>
                e.g. Smoking, diet, obesity, alcohol intake, physical activity, blood pressure and glycaemic control (HbA1c).
            </div>
        ''')
        # Append it to the ACS flowchart container
        acs_flowchart_container.children = list(acs_flowchart_container.children) + [modifiable_risk_factors_oval_box]
        state['modifiable_risk_factors_oval_box_visible'] = True




# Function to hide the new modifiable risk factors oval box
def hide_modifiable_risk_factors_oval_box():
    if state.get('modifiable_risk_factors_oval_box_visible', False):
        acs_flowchart_container.children = [
            child for child in acs_flowchart_container.children
            if not (isinstance(child, widgets.HTML) and "Identify and address modifiable risk factors" in child.value)
        ]
        state['modifiable_risk_factors_oval_box_visible'] = False



def create_drug_interaction_flowchart():
    # Add the drug interaction question if it hasn't been created yet
    if not state.get('drug_interaction_flowchart_created', False):
        # Add a vertical line before the question
        vertical_line_html = widgets.HTML('<div class="vertical-line"></div>')  # Add vertical line

        drug_interaction_question_html = widgets.VBox([
            widgets.HTML('<div style="font-size: 14px; text-align: center;">Is the person at a high risk of having a potential drug interaction or adverse effects to high-intensity statin?</div>'),
            widgets.HBox([
                drug_interaction_yes_button,
                widgets.HTML('<div class="horizontal-line"></div>'),
                drug_interaction_info_button,
                widgets.HTML('<div class="horizontal-line"></div>'),
                drug_interaction_no_button
            ], layout=widgets.Layout(justify_content='center', align_items='center', width='350px')),
            drug_interaction_info_box_container  # Info box, if needed
        ], layout=widgets.Layout(align_items='center'))

        # Add this new question to the ACS flowchart container, starting with the vertical line
        acs_flowchart_container.children = list(acs_flowchart_container.children) + [vertical_line_html, drug_interaction_question_html]
        acs_flowchart_container.children = list(acs_flowchart_container.children) + [drug_interaction_flowchart_container]

        
        
        state['drug_interaction_flowchart_created'] = True  # Set flag to prevent duplication



# Function to create additional flowchart components when "≤ 84 years" is selected


# <!-- ## 16. CKD display, diabetes display, and QRISK3 sections

# Creates CKD/diabetes branch content, age restoration, Type 2 QRISK3 entry, and related treatment sections. -->

def create_additional_flowchart():
    if not state['flowchart_created']:
        additional_line_html = widgets.HTML('<div class="vertical-line"></div>')
        additional_question_html = widgets.VBox([

            widgets.HTML('''
                <div style="font-size: 14px; text-align: center;">
                    Does this person have chronic kidney disease? 
                </div>
            '''),

            widgets.HBox([

                ckd_yes_button,
                widgets.HTML('<div class="horizontal-line"></div>'),
                ckd_info_button,
                widgets.HTML('<div class="horizontal-line"></div>'),
                ckd_no_button


            ], layout=widgets.Layout(justify_content='center', align_items='center', width='350px')),
            ckd_info_box_container,
            ckd_yes_section_container
        ], layout=widgets.Layout(align_items='center'))

        # Append the additional components to the primary flowchart container
        primary_flowchart_container.children += (additional_line_html, additional_question_html)
        state['flowchart_created'] = True

def clear_ckd_flowchart():
    children = list(primary_flowchart_container.children)
    primary_flowchart_container.children = children[:4]
    state['flowchart_created'] = False
    state['ckd_flowchart_created'] = False
    clear_ckd_yes_section()
    clear_diabetes_question()
    state['diabetes_question_visible'] = False

def clear_ckd_yes_section():
    ckd_yes_section_container.children = []





def build_target_non_hdl_section(width='420px'):
    """
    Reusable orange glowing Target Non-HDL input box
    + green follow-up box.
    Returns:
        target_non_hdl_box, follow_up_box
    """

    non_hdl_label = widgets.HTML(
        value="""
        <div style="
            font-size:15px;
            font-weight:500;
            white-space:nowrap;
            margin-right:8px;
        ">
            Baseline Non-HDL (mmol/L):
        </div>
        """
    )

    non_hdl_input = widgets.Text(
        value='',
        placeholder='e.g. 4.8',
        layout=widgets.Layout(width='90px')
    )

    target_output = widgets.HTML(
        value='',
        layout=widgets.Layout(width='100%')
    )

    non_hdl_warning = widgets.HTML(
        value='',
        layout=widgets.Layout(width='100%')
    )

    non_hdl_enter_button = widgets.Button(
        description="Enter",
        layout=widgets.Layout(width='85px', height='30px')
    )

    def calculate_target(b):
        raw_value = non_hdl_input.value.strip()

        target_output.value = ''
        non_hdl_warning.value = ''

        try:
            baseline = float(raw_value)
        except ValueError:
            non_hdl_warning.value = """
            <div style="
                color:#b00020;
                font-weight:bold;
                font-size:15px;
                text-align:center;
                margin-top:8px;
            ">
                Invalid Non-HDL. Please enter a valid number.
            </div>
            """
            return

        if baseline < 0:
            non_hdl_warning.value = """
            <div style="
                color:#b00020;
                font-weight:bold;
                font-size:15px;
                text-align:center;
                margin-top:8px;
            ">
                Invalid Non-HDL. Please enter a valid number.
            </div>
            """
            return

        target = baseline * 0.6

        target_output.value = f"""
        <div style="
            font-size:18px;
            font-weight:bold;
            text-align:center;
            margin-top:10px;
            text-decoration:underline;
        ">
            Target Non-HDL: {target:.2f} mmol/L
        </div>
        """

    non_hdl_enter_button.on_click(calculate_target)

    target_non_hdl_input_row = widgets.HBox(
        [non_hdl_label, non_hdl_input, non_hdl_enter_button],
        layout=widgets.Layout(
            justify_content='center',
            align_items='center',
            width='100%',
            margin='6px 0 2px 0',
            overflow='visible'
        )
    )

    target_non_hdl_box = widgets.VBox([
        widgets.HTML("""
        <div class="target-nonhdl-title">
            For Target Non-HDL cholesterol, please enter baseline 
            pre-treatment Non-HDL here:
        </div>
        """),
        target_non_hdl_input_row,
        non_hdl_warning,
        target_output
    ],
    layout=widgets.Layout(
        align_items='center',
        width=width
    ))

    target_non_hdl_box.add_class("target-nonhdl-box")

    follow_up_box = widgets.HTML(
        value="""
        <div class="followup-green-box">
            Measure lipid profile after 3 months and aim for the above Non-HDL cholesterol target.
        </div>
        """,
        layout=widgets.Layout(
            width=width,
            margin='12px auto'
        )
    )

    return target_non_hdl_box, follow_up_box



def display_ckd_yes_section():
    """
    CKD = Yes branch for primary prevention.
    Uses the same design style as Type 1 / Type 2 diabetes:
    - advice box
    - expandable examples for modifiable risk factors
    - expandable examples for additional risk factors
    - orange glowing target Non-HDL box
    - green follow-up box
    """

    CKD_WIDTH = '550px'

    ckd_yes_section_container.children = []

    modifiable_examples = widgets.HTML(
        value="""
        <div style="
            text-align:left;
            font-size:14px;
            line-height:1.6;
            margin-top:8px;
            padding:10px 14px;
            border:1px solid #cccccc;
            border-radius:10px;
            background-color:#f8f8f8;
        ">
            <strong>Examples:</strong>
            <ul style="margin-top:6px; margin-bottom:0;">
                <li>Smoking</li>
                <li>Diet</li>
                <li>Obesity</li>
                <li>Alcohol intake</li>
                <li>Physical activity</li>
                <li>Blood pressure</li>
                <li>HbA1c</li>
            </ul>
        </div>
        """
    )

    additional_risk_examples = widgets.HTML(
        value="""
        <div style="
            text-align:left;
            font-size:14px;
            line-height:1.6;
            margin-top:8px;
            padding:10px 14px;
            border:1px solid #cccccc;
            border-radius:10px;
            background-color:#f8f8f8;
        ">
            <strong>Examples:</strong>
            <ul style="margin-top:6px; margin-bottom:0;">
                <li>Treated for HIV</li>
                <li>Severe mental illness</li>
                <li>Taking medicines that cause dyslipidaemia</li>
                <li>Systemic inflammatory disorder</li>
                <li>Impaired fasting glycaemia</li>
            </ul>
        </div>
        """
    )

    modifiable_examples_box = widgets.VBox([])
    additional_risk_examples_box = widgets.VBox([])

    modifiable_button = widgets.Button(
        description="Show examples",
        layout=widgets.Layout(width="130px", height="28px")
    )

    additional_risk_button = widgets.Button(
        description="Show examples",
        layout=widgets.Layout(width="130px", height="28px")
    )

    def toggle_modifiable_examples(b):
        if len(modifiable_examples_box.children) == 0:
            modifiable_examples_box.children = [modifiable_examples]
            modifiable_button.description = "Hide examples"
        else:
            modifiable_examples_box.children = []
            modifiable_button.description = "Show examples"

    def toggle_additional_risk_examples(b):
        if len(additional_risk_examples_box.children) == 0:
            additional_risk_examples_box.children = [additional_risk_examples]
            additional_risk_button.description = "Hide examples"
        else:
            additional_risk_examples_box.children = []
            additional_risk_button.description = "Show examples"

    modifiable_button.on_click(toggle_modifiable_examples)
    additional_risk_button.on_click(toggle_additional_risk_examples)

    ckd_advice_box = widgets.VBox(
        [
            widgets.HTML(
                value="""
                <div style="
                    font-size:18px;
                    font-weight:bold;
                    text-align:center;
                    margin-bottom:14px;
                ">
                    CKD: Primary prevention statin treatment advice
                </div>
                """
            ),

            widgets.HBox(
                [
                    widgets.HTML(
                        value="""
                        <div style="font-size:15px; line-height:1.65;">
                            <strong>1. Discuss benefits of lifestyle changes and address all modifiable risk factors.</strong>
                        </div>
                        """,
                        layout=widgets.Layout(width="400px")
                    ),
                    modifiable_button
                ],
                layout=widgets.Layout(
                    justify_content="flex-start",
                    align_items="center",
                    width="100%",
                    gap="6px"
                )
            ),
            modifiable_examples_box,

            widgets.HBox(
                [
                    widgets.HTML(
                        value="""
                        <div style="font-size:15px; line-height:1.65; margin-top:10px;">
                            <strong>2. Consider additional risk factors if present.</strong>
                        </div>
                        """,
                        layout=widgets.Layout(width="400px")
                    ),
                    additional_risk_button
                ],
                layout=widgets.Layout(
                    justify_content="flex-start",
                    align_items="center",
                    width="100%",
                    gap="6px"
                )
            ),
            additional_risk_examples_box,

            widgets.HTML(
                value="""
                <div style="font-size:15px; line-height:1.65; margin-top:10px;">
                    <strong>3. Offer Atorvastatin 20mg once daily.</strong>
                </div>
                """
            ),
        ],
        layout=widgets.Layout(
            width=CKD_WIDTH,
            box_sizing="border-box",
            margin="12px auto",
            padding="16px 18px",
            border="2px solid black",
            border_radius="25px",
            background_color="#ffffff"
        )
    )

    target_non_hdl_box, follow_up_box = build_target_non_hdl_section(width=CKD_WIDTH)

    ckd_yes_section_container.children = [
        widgets.HTML('<div class="vertical-line"></div>'),
        widgets.HBox(
            [ckd_advice_box],
            layout=widgets.Layout(
                justify_content='center',
                width='100%'
            )
        ),
        target_non_hdl_box,
        follow_up_box
    ]



#    ckd_yes_section_container.children += [lifestyle_box, additional_risk_box, statin_box, widgets.HBox([non_hdl_input, enter_button]), target_output]

def display_diabetes_question():
    if not state['diabetes_question_visible']:
        diabetes_line_html = widgets.HTML('<div class="vertical-line"></div>')
        diabetes_question = widgets.HTML('<div style="font-size: 14px; text-align: center;">Does this person have diabetes?</div>')

        # Ensure the diabetes buttons container is centered and does not shift
        diabetes_buttons = widgets.HBox([
            diabetes_yes_button,
            widgets.HTML('<div class="horizontal-line"></div>'),
            diabetes_info_button,
            widgets.HTML('<div class="horizontal-line"></div>'),
            diabetes_no_button
        ], layout=widgets.Layout(justify_content='center', align_items='center', width='350px'))  # Set fixed width

        diabetes_question_container.children = [
            diabetes_line_html,
            diabetes_question,
            widgets.HBox([diabetes_buttons], layout=widgets.Layout(justify_content='center', width='100%')),  # Center the diabetes buttons container
            diabetes_info_box_container,
            diabetes_yes_section_container
        ]

        primary_flowchart_container.children += (diabetes_question_container,)
        state['diabetes_question_visible'] = True

# [V2 cleanup] Removed first overwritten display_diabetes_yes_section definition; later definition is the active one.

def display_diabetes_yes_section():
    if not state['diabetes_yes_section_visible']:
        diabetes_options = widgets.HBox([
            type1_button,
            type2_button
        ], layout=widgets.Layout(justify_content='center', align_items='center', width='100%'))  # Ensure full width for alignment

        diabetes_yes_section_container.children = [diabetes_options]
        state['diabetes_yes_section_visible'] = True

# Function to handle age option selection
def select_age_option_84(b, restore=False):
    state['selected_age_option'] = 'option1'
    update_age_option_styles(age_option_1)
    if not state['flowchart_created']:
        create_additional_flowchart()
        state['flowchart_created'] = True

    if state['selected_age_option'] == 'option1' and not restore:
        state['current_state'] = 'age_option_1'

    # Restore CKD state if previously selected
    if state['ckd_state']:
        restore_ckd_state()

    # Restore Type 1 diabetes state if previously selected
    if state['type1_state']:
        restore_type1_state()

    # Hide the oval box if visible
    if state['oval_box_visible']:
        toggle_oval_box(None)

def select_age_option_above_84(b, restore=False):
    state['selected_age_option'] = 'option2'
    update_age_option_styles(age_option_2)

    # Clear CKD-related components if they were previously displayed
    clear_ckd_flowchart()

    if state['selected_age_option'] == 'option2' and not restore:
        state['current_state'] = 'age_option_2'

    # Ensure the oval box is displayed
    if not state['oval_box_visible'] or restore:
        toggle_oval_box(None, force_visible=True)

# Function to toggle the oval box
def toggle_oval_box(b, force_visible=False):
    if force_visible:
        state['oval_box_visible'] = True
    else:
        state['oval_box_visible'] = not state['oval_box_visible']

    if state['oval_box_visible']:
        oval_box = widgets.HTML('<div class="expandable-section" style="text-align: center;">Consider co-morbidities, frailty and life expectancy. Offer statin if appropriate.</div>')
        age_input_container.children += (oval_box,)
    else:
        age_input_container.children = age_input_container.children[:-1]


# Function to restore CKD state
def restore_ckd_state():
    if state['ckd_state'] == 'ckd_no_button':
        on_ckd_no_button_click(None, restore=True)
    elif isinstance(state['ckd_state'], dict):
        state['ckd_yes_button'] = state['ckd_state']['yes_button']
        state['ckd_yes_section_visible'] = state['ckd_state']['yes_section_visible']
        ckd_yes_section_container.children = state['ckd_state']['additional_section']
        highlight_button([ckd_no_button, ckd_yes_button, ckd_info_button], ckd_yes_button)
        display_ckd_yes_section()

# Function to restore Type 1 diabetes state
def restore_type1_state():
    if state['type1_state']:
        state['type1_selected'] = state['type1_state']['type1_selected']
        state['type2_selected'] = state['type1_state']['type2_selected']
        state['type1_options_visible'] = state['type1_state']['type1_options_visible']
        state['type1_any_yes'] = state['type1_state']['type1_any_yes']
        on_type1_button_click(None)
        if state['type1_any_yes']:
            display_additional_section()

def on_qrisk3_value_change(change):
    warning_message.value = ''
    clear_type2_additional_section()
    
def on_qrisk3_enter(b):
    score = qrisk3_input.value

    # Always clear previous QRISK3 ≥10 advice sections first.
    # This prevents old advice/target boxes staying visible after a new invalid or <10 value.
    clear_type2_additional_section()

    if score is None:
        warning_message.value = """
        <div style="
            color:#b00020;
            font-weight:bold;
            font-size:15px;
            text-align:center;
            margin-top:8px;
        ">
            Invalid QRISK3. Please enter a valid percentage.
        </div>
        """
        return

    if score < 0 or score > 100:
        warning_message.value = """
        <div style="
            color:#b00020;
            font-weight:bold;
            font-size:15px;
            text-align:center;
            margin-top:8px;
        ">
            Invalid QRISK3. Please enter a valid percentage.
        </div>
        """
        return

    qrisk3_result.value = ''
    warning_message.value = ''



    if score < QRISK_THRESHOLDS["OFFER_STATIN_PERCENT"]:
        qrisk3_result.value = f"""
        <div class="oval-box" style="
            background-color:#f2f2f2;
            font-size:15px;
            line-height:1.5;
            width:{QRISK3_SECTION_WIDTH};
            box-sizing:border-box;
            margin:14px auto;
            text-align:center;
        ">
            <b>Type 2 diabetes and QRISK3 &lt; {QRISK_THRESHOLDS["OFFER_STATIN_PERCENT"]}%</b><br><br>

            Discuss lifestyle modification and optimise modifiable cardiovascular risk factors,
            including smoking, diet, weight management, physical activity, blood pressure control
            and glycaemic control.<br><br>

            Do not rule out treatment with Atorvastatin 20 mg for primary prevention just because
            QRISK3 is below {QRISK_THRESHOLDS["OFFER_STATIN_PERCENT"]}% if the person has an
            informed preference for taking a statin, or if there are concerns that cardiovascular
            risk may be underestimated.<br><br>

            <a href="{CLINICAL_LINKS["NICE_NG238"]}"
               target="_blank"
               style="
                   color:#0066cc;
                   font-weight:bold;
                   text-decoration:underline;
               ">
               View NICE NG238 lipid modification guidance
            </a>
        </div>
        """
        return

    if score >= QRISK_THRESHOLDS["OFFER_STATIN_PERCENT"]:
        display_type2_additional_section()










# Function to display QRISK3 section for Type 2 diabetes
QRISK3_SECTION_WIDTH = '420px'
warning_message = widgets.HTML(
    value='',
    layout=widgets.Layout(width=QRISK3_SECTION_WIDTH)
)
def display_type2_qrisk3_section():
    global qrisk3_box, qrisk3_result, enter_button, qrisk3_input
    qrisk3_input = widgets.FloatText(
        description='QRISK3 score (%):',
        layout=widgets.Layout(width='300px'),
        style={'description_width': 'initial', 'font_size': '18px'}  # Adjust font size
    )
    enter_button = widgets.Button(description="Enter", layout=widgets.Layout(width='100px', height='30px'))
    enter_button.on_click(on_qrisk3_enter)


    qrisk3_input_row = widgets.HBox(
        [qrisk3_input_row, enter_button],
        layout=widgets.Layout(
            justify_content='center',
            align_items='center',
            width='100%'
        )
    )
    
    qrisk3_input_row.add_class("qrisk3-input-row")
    
    qrisk3_box = widgets.VBox([
        widgets.HTML('<div class="option-title" style="text-align: center;"><strong>Calculate QRISK3 score:</strong></div>'),
        widgets.HTML(
    f'<div class="option-title" style="text-align: center;"><a href="{CLINICAL_LINKS["QRISK3"]}" target="_blank">{CLINICAL_LINKS["QRISK3"]}</a></div>'
),
        

        
        widgets.HBox([qrisk3_input_row, enter_button], layout=widgets.Layout(justify_content='center'))

    ], layout=widgets.Layout(
        align_items='center',
        width='100%',
        padding='10px',
        border='2px solid black',
        border_radius='8px',
        background_color='#f9f9f9',
        font_size='18px',
        margin='10px 0 0 0'
    ))

    # Placeholder for the result message
    qrisk3_result = widgets.HTML(value='', layout=widgets.Layout(width='300px', style={'font_size': '18px'}))

    # Add the result message and warning message below the QRISK3 section
    diabetes_yes_section_container.children = [widgets.HBox([type1_button, type2_button], layout=widgets.Layout(justify_content='center', width='100%')), qrisk3_box, qrisk3_result, warning_message]



def display_qrisk3_section(show_diabetes_type_buttons=True):
    global qrisk3_box, qrisk3_result, enter_button, qrisk3_input

    qrisk3_label = widgets.HTML(
        value="""
        <div style="
            font-size:14px;
            font-weight:bold;
            line-height:1.4;
        ">
            Enter QRISK3 result (%):
        </div>
        """
    )

    qrisk3_input = widgets.FloatText(
        value=0,
        step=0.1,
        description='',
        layout=widgets.Layout(width='80px')
    )

    qrisk3_input.add_class("qrisk3-input-row")
    qrisk3_input.observe(on_qrisk3_value_change, names='value')

    enter_button = widgets.Button(
        description="Enter",
        layout=widgets.Layout(width='100px', height='30px')
    )

    enter_button.on_click(on_qrisk3_enter)

    qrisk3_input_row = widgets.HBox(
        [qrisk3_label, qrisk3_input, enter_button],
        layout=widgets.Layout(
            justify_content='center',
            align_items='center',
            width='100%',
            gap='8px'
        )
    )

    qrisk3_input_row.add_class("qrisk3-input-row")
    

    qrisk3_box = widgets.VBox([
        widgets.HTML(
            '<div class="option-title" style="text-align: center;">'
            '<strong>Calculate QRISK3 score</strong>'
            '</div>'
        ),

        widgets.HTML("""
        <div style="
            text-align:center;
            padding:12px;
            border:2px solid #0066cc;
            border-radius:10px;
            background-color:#e8f4ff;
            margin:10px;
        ">
            <b>Step 1: Open the QRISK3 calculator</b> 
            <br><br>

            <a href="{CLINICAL_LINKS["QRISK3"]}"
               target="_blank"
               class="qrisk-link"
               style="
                   display:inline-block;
                   padding:12px 24px;
                   background-color:#0066cc;
                   color:white !important;
                   text-decoration:none;
                   border-radius:10px;
                   font-weight:bold;
                   font-size:18px;
                   animation: qriskPulse 1.8s ease-in-out infinite;
               ">
               <span style="color:white !important;">
                   Open QRISK3 Calculator
               </span>
            </a>
            <br><br>
            <b>Step 2: Calculate the QRISK3 score</b>
            <br>
            <b>Step 3: Return to this tool and enter the result below</b>
        </div>
        """),

        qrisk3_input_row,

    ],
    layout=widgets.Layout(
        align_items='center',
        width=QRISK3_SECTION_WIDTH,
        padding='10px',
        border='2px solid black',
        border_radius='8px',
        background_color='#f9f9f9',
        font_size='18px'
    ))

    qrisk3_result = widgets.HTML(
        value='',
        layout=widgets.Layout(width='300px')
    )

    qrisk3_children = []
    
    if show_diabetes_type_buttons:
        qrisk3_children.append(
            widgets.HBox(
                [type1_button, type2_button],
                layout=widgets.Layout(justify_content='center', width='100%', margin='0 0 20px 0')
            )
        )
    
    
    qrisk3_children += [
        widgets.HBox(
            [qrisk3_box],
            layout=widgets.Layout(
                justify_content='center',
                width='100%',
                margin='15px 0 0 0'
            )
        ),
        qrisk3_result,
        warning_message
    ]
        
    diabetes_yes_section_container.children = qrisk3_children

# Function to clear additional section for QRISK3 score < 10%
def clear_type2_additional_section():
    children_to_keep = []

    for child in diabetes_yes_section_container.children:

        is_qrisk3_advice_box = (
            (
                isinstance(child, widgets.HTML)
                and f'QRISK3 ≥ {QRISK_THRESHOLDS["OFFER_STATIN_PERCENT"]}%' in child.value
            )
            or
            (
                isinstance(child, widgets.VBox)
                and any(
                    isinstance(grandchild, widgets.HTML)
                    and f'QRISK3 ≥ {QRISK_THRESHOLDS["OFFER_STATIN_PERCENT"]}%' in grandchild.value
                    for grandchild in child.children
                )
            )
            or
            (
                isinstance(child, widgets.HBox)
                and any(
                    isinstance(grandchild, widgets.VBox)
                    and any(
                        isinstance(great_grandchild, widgets.HTML)
                        and f'QRISK3 ≥ {QRISK_THRESHOLDS["OFFER_STATIN_PERCENT"]}%' in great_grandchild.value
                        for great_grandchild in grandchild.children
                    )
                    for grandchild in child.children
                )
            )
        )


        is_target_non_hdl_box = (
            isinstance(child, widgets.VBox)
            and "target-nonhdl-box" in getattr(child, "_dom_classes", ())
        )

        is_follow_up_box = (
            isinstance(child, widgets.HTML)
            and "Measure lipid profile after 3 months" in child.value
        )

        if (
            not is_qrisk3_advice_box
            and not is_target_non_hdl_box
            and not is_follow_up_box
        ):
            children_to_keep.append(child)

    diabetes_yes_section_container.children = children_to_keep









def display_type2_additional_section():
    # Check if the additional section is already present

    if not any(
        isinstance(child, widgets.HTML)
        and f'QRISK3 ≥ {QRISK_THRESHOLDS["OFFER_STATIN_PERCENT"]}%' in child.value
        for child in diabetes_yes_section_container.children
    ):


    
        modifiable_examples = widgets.HTML(
            value="""
            <div style="
                text-align:left;
                font-size:14px;
                line-height:1.6;
                margin-top:8px;
                padding:10px 14px;
                border:1px solid #cccccc;
                border-radius:10px;
                background-color:#f8f8f8;
            ">
                <strong>Examples:</strong>
                <ul style="margin-top:6px; margin-bottom:0;">
                    <li>Smoking</li>
                    <li>Diet</li>
                    <li>Obesity</li>
                    <li>Alcohol intake</li>
                    <li>Physical activity</li>
                    <li>Blood pressure</li>
                    <li>Glycaemic control / HbA1c</li>
                </ul>
            </div>
            """
        )
    
        additional_risk_examples = widgets.HTML(
            value="""
            <div style="
                text-align:left;
                font-size:14px;
                line-height:1.6;
                margin-top:8px;
                padding:10px 14px;
                border:1px solid #cccccc;
                border-radius:10px;
                background-color:#f8f8f8;
            ">
                <strong>Examples:</strong>
                <ul style="margin-top:6px; margin-bottom:0;">
                    <li>Treated for HIV</li>
                    <li>Severe mental illness</li>
                    <li>Taking medicines that cause dyslipidaemia</li>
                    <li>Systemic inflammatory disorder</li>
                    <li>Impaired fasting glycaemia</li>
                </ul>
            </div>
            """
        )
    
        modifiable_examples_box = widgets.VBox([])
        additional_risk_examples_box = widgets.VBox([])
    
        modifiable_button = widgets.Button(
            description="Show examples",
            layout=widgets.Layout(width="130px", height="28px")
        )
    
        additional_risk_button = widgets.Button(
            description="Show examples",
            layout=widgets.Layout(width="130px", height="28px")
        )
    
        def toggle_modifiable_examples(b):
            if len(modifiable_examples_box.children) == 0:
                modifiable_examples_box.children = [modifiable_examples]
                modifiable_button.description = "Hide examples"
            else:
                modifiable_examples_box.children = []
                modifiable_button.description = "Show examples"
    
        def toggle_additional_risk_examples(b):
            if len(additional_risk_examples_box.children) == 0:
                additional_risk_examples_box.children = [additional_risk_examples]
                additional_risk_button.description = "Hide examples"
            else:
                additional_risk_examples_box.children = []
                additional_risk_button.description = "Show examples"
    
        modifiable_button.on_click(toggle_modifiable_examples)
        additional_risk_button.on_click(toggle_additional_risk_examples)

        
    
        advice_box = widgets.VBox(
            [
                widgets.HTML(
                    value="""
                    <div style="
                        font-size:18px;
                        font-weight:bold;
                        text-align:center;
                        margin-bottom:14px;
                    ">
                        QRISK3 ≥ 10%: Primary prevention statin treatment advice
                    </div>
                    """
                ),
    
                widgets.HBox(
                    [
                        widgets.HTML(
                            value="""
                            <div style="font-size:15px; line-height:1.65;">
                                <strong>1. Discuss benefits of lifestyle changes and address all modifiable risk factors.</strong>
                            </div>
                            """,
                            layout=widgets.Layout(width="330px")
                        ),
                        modifiable_button
                    ],
                    layout=widgets.Layout(
                        justify_content="flex-start",
                        align_items="center",
                        width="100%",
                        gap="6px"
                    )
                ),
                modifiable_examples_box,
    
                widgets.HBox(
                    [
                        widgets.HTML(
                            value="""
                            <div style="font-size:15px; line-height:1.65; margin-top:10px;">
                                <strong>2. Consider additional risk factors if present.</strong>
                            </div>
                            """,
                            layout=widgets.Layout(width="330px")
                        ),
                        additional_risk_button
                    ],
                    layout=widgets.Layout(
                        justify_content="flex-start",
                        align_items="center",
                        width="100%",
                        gap="6px"
                    )
                ),
                additional_risk_examples_box,
    
                widgets.HTML(
                    value="""
                    <div style="font-size:15px; line-height:1.65; margin-top:10px;">
                        <strong>3. Offer Atorvastatin 20mg once daily.</strong>
                    </div>
                    """
                ),
            ],
            layout=widgets.Layout(
                width="550px",
                box_sizing="border-box",
                margin="12px auto",
                padding="16px 18px",
                border="2px solid black",
                border_radius="25px",
                background_color="#ffffff"
            )
        )


        global non_hdl_input, target_output

        non_hdl_label = widgets.HTML(
            value="""
            <div style="
                font-size:15px;
                font-weight:500;
                white-space:nowrap;
                margin-right:8px;
            ">
                Baseline Non-HDL (mmol/L):
            </div>
            """
        )
        
        non_hdl_input = widgets.Text(
            value='',
            placeholder='e.g. 4.8',
            layout=widgets.Layout(width='90px')
        )

        target_output = widgets.HTML(
            value='',
            layout=widgets.Layout(width='100%')
        )

        non_hdl_warning = widgets.HTML(
            value='',
            layout=widgets.Layout(width='100%')
        )

        non_hdl_enter_button = widgets.Button(
            description="Enter",
            layout=widgets.Layout(width='85px', height='30px')
        )

        def calculate_target(b):
            raw_value = non_hdl_input.value.strip()

            target_output.value = ''
            non_hdl_warning.value = ''

            try:
                baseline = float(raw_value)
            except ValueError:
                non_hdl_warning.value = """
                <div style="
                    color:#b00020;
                    font-weight:bold;
                    font-size:15px;
                    text-align:center;
                    margin-top:8px;
                ">
                    Invalid Non-HDL. Please enter a valid number.
                </div>
                """
                return

            if baseline < 0:
                non_hdl_warning.value = """
                <div style="
                    color:#b00020;
                    font-weight:bold;
                    font-size:15px;
                    text-align:center;
                    margin-top:8px;
                ">
                    Invalid Non-HDL. Please enter a valid number.
                </div>
                """
                return

            target = baseline * 0.6

            target_output.value = f"""
            <div style="
                font-size:18px;
                font-weight:bold;
                text-align:center;
                margin-top:10px;
                text-decoration:underline;
            ">
                Target Non-HDL: {target:.2f} mmol/L
            </div>
            """

        non_hdl_enter_button.on_click(calculate_target)

        target_non_hdl_input_row = widgets.HBox(
            [non_hdl_label, non_hdl_input, non_hdl_enter_button],
            layout=widgets.Layout(
                justify_content='center',
                align_items='center',
                width='100%',
                margin='6px 0 2px 0',
                overflow='visible'
            )
        )

        target_non_hdl_box = widgets.VBox([
            widgets.HTML("""
            <div class="target-nonhdl-title">
                For Target Non-HDL cholesterol, please enter baseline 
                pre-treatment Non-HDL here:
            </div>
            """),
            target_non_hdl_input_row,
            non_hdl_warning,
            target_output
        ],
        layout=widgets.Layout(
            align_items='center',
            width=QRISK3_SECTION_WIDTH
        ))

        target_non_hdl_box.add_class("target-nonhdl-box")
        
        follow_up_box = widgets.HTML(
            value="""
            <div class="followup-green-box">
                Measure lipid profile after 3 months and aim for the above Non-HDL cholesterol target.
            </div>
            """,
            layout=widgets.Layout(
                width=QRISK3_SECTION_WIDTH,
                margin='12px auto'
            )
        )
        
        diabetes_yes_section_container.children += (
            widgets.HBox(
                [advice_box],
                layout=widgets.Layout(
                    justify_content='center',
                    width='100%'
                )
            ),
            target_non_hdl_box,
            follow_up_box
        )


<!-- ## 17. Non-HDL cholesterol branch

Creates, restores, clears, and handles the Non-HDL cholesterol branch under Secondary Prevention. -->

In [18]:
def create_non_hdl_flowchart():
    if state.get('non_hdl_flowchart_created', False):
       # print("Non-HDL flowchart already exists. Skipping creation.")
        return

    # Add checks to ensure prerequisites are met (e.g., Drug Interaction is completed)
    if not state.get('drug_interaction_yes_button', False) and not state.get('drug_interaction_no_button', False):
       # print("Cannot create Non-HDL flowchart: Drug Interaction flowchart not completed.")
        return

   # print("Creating Non-HDL flowchart...")

    # # Clear any existing Non-HDL elements
    # clear_non_hdl_elements()

    # Add the Non-HDL cholesterol question
    non_hdl_question_html = widgets.VBox([
        widgets.HTML(
            f'<div style="font-size: 14px; text-align: center;">'
            f'Is Non-HDL cholesterol less than '
            f'{NON_HDL_TREATMENT_THRESHOLDS["SECONDARY_TARGET"]} mmol/L?'
            f'</div>'
        ),

        widgets.HBox([
            non_hdl_yes_button,
            widgets.HTML('<div class="horizontal-line"></div>'),
            non_hdl_info_button,
            widgets.HTML('<div class="horizontal-line"></div>'),
            non_hdl_no_button
        ], layout=widgets.Layout(justify_content='center', align_items='center', width='350px')),
        non_hdl_info_box_container
    ], layout=widgets.Layout(align_items='center'))



    # Add new Non-HDL flowchart components
    vertical_line_html = widgets.HTML('<div class="vertical-line"></div>')
    drug_interaction_flowchart_container.children += (
        vertical_line_html,
        non_hdl_question_html,
        non_hdl_flowchart_container
    )

    # Update state
    state['non_hdl_flowchart_created'] = True

def restore_non_hdl():
    """Restore the Non-HDL flowchart state."""
    # Check if the Non-HDL flowchart was previously created
    if state.get('non_hdl_flowchart_created', False):
        create_non_hdl_flowchart()  # Recreate the Non-HDL flowchart

        # Restore the selected state (Yes or No)
        if state.get('non_hdl_yes_button', False):
            on_non_hdl_yes_button_click(None, restore=True)
        elif state.get('non_hdl_no_button', False):
            on_non_hdl_no_button_click(None, restore=True)
    else:
        # If the Non-HDL flowchart was not created, ensure it's cleared
        clear_non_hdl_flowchart()


def update_non_hdl_state():
    """Restore or clear the Non-HDL flowchart based on the previous state."""
    if state.get('non_hdl_flowchart_created', False):
        # Restore Non-HDL flowchart and its buttons
        create_non_hdl_flowchart()  # Recreate the Non-HDL flowchart
        if state.get('non_hdl_yes_button', False):
            on_non_hdl_yes_button_click(None, restore=True)  # Restore "Yes" state
        elif state.get('non_hdl_no_button', False):
            on_non_hdl_no_button_click(None, restore=True)  # Restore "No" state
    else:
        # Clear the Non-HDL flowchart if it was not previously created
        clear_non_hdl_flowchart()

# Clear the Non-HDL flowchart

def clear_non_hdl_flowchart():

    drug_interaction_flowchart_container.children = [
        child for child in drug_interaction_flowchart_container.children
        if not (
            isinstance(child, widgets.VBox) and any(
                isinstance(grandchild, widgets.HTML)
                and NON_HDL_TARGET_TEXT in grandchild.value
                for grandchild in child.children
            )
        )
    ]

    state['non_hdl_flowchart_created'] = False



# Handler for the Non-HDL "Yes" button


def on_non_hdl_yes_button_click(b, restore=False):
   # print("Non-HDL Yes button clicked.")
    state['non_hdl_yes_button'] = True
    state['non_hdl_no_button'] = False
    highlight_button([non_hdl_yes_button, non_hdl_no_button, non_hdl_info_button], non_hdl_yes_button)

    if not restore:
        state['current_state'] = 'non_hdl_yes_button'
    
    # Reset the flag so that the Triglycerides branch is recreated
    state['triglycerides_flowchart_created'] = False
    create_triglycerides_flowchart()



def on_non_hdl_no_button_click(b, restore=False):
   # print("Non-HDL No button clicked.")
    state['non_hdl_yes_button'] = False
    state['non_hdl_no_button'] = True
    highlight_button([non_hdl_yes_button, non_hdl_no_button, non_hdl_info_button], non_hdl_no_button)

    if not restore:
        state['current_state'] = 'non_hdl_no_button'

    # Create a new vertical line and the Atorvastatin branch
    vertical_line_new = widgets.HTML('<div class="vertical-line"></div>')

    increase_oval = widgets.HTML('''
        <div class="oval-box" style="
            padding:8px 20px;
            border:2px solid black;
            border-radius:25px;
            background-color:#FFFFFF;
            font-size:13px;
            margin:10px;
            text-align:center;">
            <strong>Increase to Atorvastatin 80mg daily</strong>
        </div>
    ''')

    atorvastatin_question = widgets.VBox([
        widgets.HTML(
            '<div style="font-size: 14px; text-align: center;">'
            'Is this person already on Atorvastatin 80 mg daily?</div>'
        ),
        widgets.HBox([
            atorvastatin_yes_button,
            widgets.HTML('<div class="horizontal-line"></div>'),
            atorvastatin_info_button,
            widgets.HTML('<div class="horizontal-line"></div>'),
            atorvastatin_no_button
        ], layout=widgets.Layout(justify_content='center', align_items='center', width='350px'))
    ], layout=widgets.Layout(align_items='center'))

    # Create (or update) a container for the Atorvastatin branch
    global atorvastatin_flowchart_container
    atorvastatin_flowchart_container = widgets.VBox(
        [vertical_line_new, increase_oval, atorvastatin_question],
        layout=widgets.Layout(align_items='center')
    )

    # Replace the current children of non_hdl_flowchart_container with the new branch.
    non_hdl_flowchart_container.children = (atorvastatin_flowchart_container,)




# Info box toggle for Non-HDL Question
def toggle_non_hdl_info_box(b):
    state['non_hdl_info_box_visible'] = not state['non_hdl_info_box_visible']
    if state['non_hdl_info_box_visible']:
        non_hdl_info_box = widgets.HTML('''
        <div class="info-box">
            
        </div>
        ''')
        non_hdl_info_box_container.children = [non_hdl_info_box]
    else:
        non_hdl_info_box_container.children = []


<!-- ## 18. Triglycerides and LDL-C / Icosapent branch

Controls triglycerides routing and LDL-C question logic leading to Icosapent or continuation advice. -->

In [19]:
def on_triglycerides_yes_button_click(b, restore=False):
   # print("Triglycerides Yes button clicked.")
    state['triglycerides_yes_button'] = True
    state['triglycerides_no_button'] = False

    highlight_button(
        [triglycerides_yes_button, triglycerides_no_button, triglycerides_info_button],
        triglycerides_yes_button
    )
    
    # Clear any existing content in the Triglycerides container
    triglycerides_flowchart_container.children = []
    
    # New oval: measure fasting lipid profile
    fasting_lipid_oval = widgets.HTML("""
    <div class="oval-box" style="
        width:350px;
        box-sizing:border-box;
        padding:14px 18px;
        border:2px solid black;
        border-radius:25px;
        background-color:white;
        margin:10px auto;
        text-align:center;
        font-size:13px;
    ">
        <strong>Measure fasting lipid profile</strong>
    </div>
    """)

    # Create a vertical line to go above the LDL question
    ldl_line = widgets.HTML('<div class="vertical-line"></div>')
    
    # Create the LDL question itself
    ldl_question_html = widgets.VBox([



        widgets.HTML(
            f'<div style="font-size: 14px; text-align: center;">'
            f'Is LDL-cholesterol '
            f'{LDL_C_TREATMENT_THRESHOLDS["ICOSAPENT_MIN"]}–{LDL_C_TREATMENT_THRESHOLDS["ICOSAPENT_MAX"]} mmol/L?'
            f'</div>'
        ),

        widgets.HBox([
            ldl_yes_button,
            widgets.HTML('<div class="horizontal-line"></div>'),
            ldl_info_button,
            widgets.HTML('<div class="horizontal-line"></div>'),
            ldl_no_button
        ], layout=widgets.Layout(justify_content='center', align_items='center', width='350px'))
    ], layout=widgets.Layout(align_items='center'))
    
    # Put the oval, vertical line, and LDL question together
    ldl_question_box = widgets.VBox(
        [fasting_lipid_oval, ldl_line, ldl_question_html],
        layout=widgets.Layout(align_items='center')
    )
    
    # Add the oval + LDL question + LDL output container
    triglycerides_flowchart_container.children += (
        ldl_question_box,
        ldl_flowchart_container
    )


def toggle_triglycerides_info_box(b):
    state['triglycerides_info_box_visible'] = not state.get('triglycerides_info_box_visible', False)
    if state['triglycerides_info_box_visible']:
        triglycerides_info_box = widgets.HTML('''
        <div class="info-box">
            Triglycerides are a type of fat found in your blood. Levels higher than 1.7 mmol/L may indicate elevated cardiovascular risk.
        </div>
        ''')
        triglycerides_info_box_container.children = [triglycerides_info_box]
    else:
        triglycerides_info_box_container.children = []

# Function to create the Triglycerides flowchart


def create_triglycerides_flowchart():
    """Create the Triglycerides flowchart elements."""
    # Check if the Triglycerides flowchart is already created
    if state.get('triglycerides_flowchart_created', False):
    #    print("Triglycerides flowchart already exists. Skipping creation.")
        return

    # Create the flowchart components
    vertical_line_html = widgets.HTML('<div class="vertical-line"></div>')
    triglycerides_question_html = widgets.VBox([
        widgets.HTML('<div style="font-size: 14px; text-align: center;">Are Triglycerides <strong>>1.7mmol/L?</strong></div>'),
        widgets.HBox([
            triglycerides_yes_button,
            widgets.HTML('<div class="horizontal-line"></div>'),
            triglycerides_info_button,
            widgets.HTML('<div class="horizontal-line"></div>'),
            triglycerides_no_button
        ], layout=widgets.Layout(justify_content='center', align_items='center', width='350px')),
        triglycerides_info_box_container
    ], layout=widgets.Layout(align_items='center'))

    # Update the flowchart container

    non_hdl_flowchart_container.children = [vertical_line_html, triglycerides_question_html, triglycerides_flowchart_container]
    state['triglycerides_flowchart_created'] = True

  #  print("Triglycerides flowchart created successfully.")


def clear_triglycerides_flowchart():
    """Clear the Triglycerides flowchart."""
    # Remove any existing Triglycerides-related elements
    non_hdl_flowchart_container.children = []
    state['triglycerides_flowchart_created'] = False
#    print("Triglycerides flowchart cleared successfully.")





def toggle_triglycerides_flowchart():
    """Toggle the Triglycerides flowchart."""
    if state.get('triglycerides_flowchart_created', False):
#        print("Toggling off Triglycerides flowchart.")
        clear_flowchart_container(triglycerides_flowchart_container)
        state['triglycerides_flowchart_created'] = False
    else:
#        print("Toggling on Triglycerides flowchart.")
        create_triglycerides_flowchart()





def toggle_non_hdl_flowchart():
    if state['non_hdl_yes_button']:
        # Recreate the Triglycerides flowchart if "Yes" is selected
        create_triglycerides_flowchart()
    elif state['non_hdl_no_button']:
        # Clear the container if "No" is selected
        non_hdl_flowchart_container.children = []
        state['triglycerides_flowchart_created'] = False





def on_ldl_yes_button_click(b, restore=False):
    # Highlight the Yes button in yellow
    highlight_button([ldl_yes_button, ldl_no_button], ldl_yes_button)
    
    # Show the "Commence Icosapent" oval


    icosapent_box = widgets.HTML("""
    <div class="oval-box">
    
        <div style="
            font-size:16px;
            font-weight:bold;
            margin-bottom:15px;
            text-align:center;
        ">
            Commence Icosapent Ethyl (Vazkepa)
        </div>
    
        <a href="{CLINICAL_LINKS["NICE_TA805_ICOSAPENT"]}"
           target="_blank"
           class="qrisk-link"
           style="
               display:inline-block;
               padding:12px 24px;
               background-color:#0066cc;
               color:white !important;
               text-decoration:none;
               border-radius:10px;
               font-weight:bold;
               font-size:18px;
               animation: qriskPulse 1.8s ease-in-out infinite;
           ">
           <span style="color:white !important;">
               Open NICE TA
           </span>
        </a>
    
    </div>
    """)
        
    ldl_flowchart_container.children = (icosapent_box,)


def on_ldl_no_button_click(b, restore=False):
    # Highlight the No button in yellow
    highlight_button([ldl_yes_button, ldl_no_button], ldl_no_button)
    
    # Show the "Continue current therapy" oval
    continue_oval = widgets.HTML('''
        <div class="oval-box" style="
            padding: 20px 40px;
            border: 2px solid black;
            border-radius: 25px;
            background-color: #FFFFFF;
            font-size: 13px;
            margin: 10px;
            text-align: center;">
            <strong>Continue current lipid-lowering therapy</strong>
        </div>
    ''')
    ldl_flowchart_container.children = (continue_oval,)



def on_triglycerides_no_button_click(b, restore=False):
#    print("Triglycerides No button clicked.")
    state['triglycerides_yes_button'] = False
    state['triglycerides_no_button'] = True
    highlight_button([triglycerides_yes_button, triglycerides_no_button, triglycerides_info_button], triglycerides_no_button)
    
    # Clear any existing content in the flowchart container
    triglycerides_flowchart_container.children = []
    
    continue_oval = widgets.HTML('''
        <div class="oval-box" style="
            padding: 20px 40px;
            border: 2px solid black;
            border-radius: 25px;
            background-color: #FFFFFF;
            font-size: 13px;
            margin: 10px;
            text-align: center;">
            Continue current lipid-lowering therapy
        </div>
    ''')
    triglycerides_flowchart_container.children += (continue_oval,)


<!-- ## 19. Atorvastatin branch selector

Tracks whether the patient is already on atorvastatin 80 mg and rebuilds the relevant sub-branch. -->

In [20]:
# Use a state variable to track the Atorvastatin choice:
state['atorvastatin_choice'] = None  # will be set to 'yes' or 'no'

def update_atorvastatin_branch():
    """Rebuild the Atorvastatin branch based on state['atorvastatin_choice']."""
    # Build the common header (always shown)
    header = (
        widgets.HTML('<div class="vertical-line"></div>'),
        widgets.VBox([
            widgets.HTML(
                '<div style="font-size:14px;text-align:center;">'
                'Is this person already on Atorvastatin 80 mg daily?</div>'
            ),
            widgets.HBox([
                atorvastatin_yes_button,
                widgets.HTML('<div class="horizontal-line"></div>'),
                atorvastatin_info_button,
                widgets.HTML('<div class="horizontal-line"></div>'),
                atorvastatin_no_button
            ], layout=widgets.Layout(justify_content='center', align_items='center', width='350px'))
        ], layout=widgets.Layout(align_items='center'))
    )
    
    # Build the sub-branch based on the current state:
    if state['atorvastatin_choice'] == 'yes':
        # Tolerance branch
        tolerance_branch = (
            widgets.HTML('<div class="vertical-line"></div>'),
            widgets.VBox([
                widgets.HTML(
                    '<div style="font-size:14px;text-align:center;">'
                    'Is patient tolerating Atorvastatin 80mg daily without significant adverse side-effects?</div>'
                ),
                widgets.HBox([
                    atorvastatin_tolerance_yes_button,
                    widgets.HTML('<div class="horizontal-line"></div>'),
                    atorvastatin_tolerance_info_button,
                    widgets.HTML('<div class="horizontal-line"></div>'),
                    atorvastatin_tolerance_no_button
                ], layout=widgets.Layout(justify_content='center', align_items='center', width='350px'))
            ], layout=widgets.Layout(align_items='center'))
        )
        # Reset the container's children to header + tolerance branch
        atorvastatin_flowchart_container.children = header + tolerance_branch
        

    elif state['atorvastatin_choice'] == 'no':
        # Increase branch
        increase_oval = widgets.HTML('''
            <div class="oval-box" style="
                padding:20px 40px;
                border:2px solid black;
                border-radius:25px;
                background-color:#FFFFFF;
                font-size:13px;
                margin:10px;
                text-align:center;">
                <strong>Increase to Atorvastatin 80mg daily</strong>
            </div>
        ''')
    
        # After increasing to Atorvastatin 80mg, continue to the tolerance question.
        tolerance_branch = (
            widgets.HTML('<div class="vertical-line"></div>'),
            increase_oval,
            widgets.HTML('<div class="vertical-line"></div>'),
            widgets.VBox([
                widgets.HTML(
                    '<div style="font-size:14px;text-align:center;">'
                    'Is patient tolerating Atorvastatin 80mg daily without significant adverse side-effects?</div>'
                ),
                widgets.HBox([
                    atorvastatin_tolerance_yes_button,
                    widgets.HTML('<div class="horizontal-line"></div>'),
                    atorvastatin_tolerance_info_button,
                    widgets.HTML('<div class="horizontal-line"></div>'),
                    atorvastatin_tolerance_no_button
                ], layout=widgets.Layout(
                    justify_content='center',
                    align_items='center',
                    width='350px'
                ))
            ], layout=widgets.Layout(align_items='center'))
        )
    
        atorvastatin_flowchart_container.children = header + tolerance_branch
    
    else:
        # If no choice yet, show only the header.
        atorvastatin_flowchart_container.children = header

def on_atorvastatin_tolerance_yes_button_click(b=None, restore=False, q8_first=False):

    # Only highlight Yes when this is the true Q6 Yes pathway.
    # When q8_first=True, this function is being reused after Q6 No.
    if not q8_first:
        highlight_button(
            [
                atorvastatin_tolerance_yes_button,
                atorvastatin_tolerance_no_button,
                atorvastatin_tolerance_info_button
            ],
            atorvastatin_tolerance_yes_button
        )

    # Keep the Atorvastatin tolerance question/buttons visible.
    # Remove only downstream content before rebuilding the branch.
    base_len = 6 if state.get('atorvastatin_choice') == 'no' else 4
    
    if len(atorvastatin_flowchart_container.children) > base_len:
        atorvastatin_flowchart_container.children = atorvastatin_flowchart_container.children[:base_len]
    
    
    

    
    # ------------------------------------------------------------
    # LDL-C input row
    # ------------------------------------------------------------
    ldl_label = widgets.HTML(
        value="<span style='font-size:16px; font-weight:bold;'>Enter LDL-C (mmol/L):</span>",
        layout=widgets.Layout(width='auto')
    )

    ldl_input = widgets.BoundedFloatText(
        value=0.0,
        min=0,
        max=20,
        step=0.1,
        description='',
        layout=widgets.Layout(width='120px')
    )
    
    ldl_input.add_class("pulse-ldl")

    ldl_button = widgets.Button(
        description="Enter",
        layout=widgets.Layout(width='100px', height='30px')
    )

    ldl_result_container = widgets.VBox(
        [],
        layout=widgets.Layout(align_items='center', width='100%')
    )

    ldl_input_box = widgets.HBox(
        [ldl_label, ldl_input, ldl_button],
        layout=widgets.Layout(
            justify_content='center',
            align_items='center',
            margin='10px 0'
        )
    )

    # ------------------------------------------------------------
    # Sampson / fasting lipid profile section
    # ------------------------------------------------------------
    sampson_section = widgets.VBox([

        widgets.HTML('''
            <div class="oval-box" style="
                width:250px;
                box-sizing:border-box;
                padding:14px 18px;
                border:2px solid black;
                border-radius:25px;
                background-color:#FFFFFF;
                font-size:13px;
                margin:10px auto;
                text-align:center;">
                <span style="font-size:18px;"><b><u>Option A:</u></b></span><br>
                <strong>Measure a fasting lipid profile<br>to obtain LDL-cholesterol value</strong>
            </div>
        '''),

        widgets.HTML('''
            <div style="
                font-size:20px;
                font-weight:bold;
                text-align:center;
                margin:4px 0;">
                OR
            </div>
        '''),

        widgets.HTML('''
            <div class="oval-box" style="
                width:420px;
                box-sizing:border-box;
                padding:14px 18px;
                border:2px solid black;
                border-radius:25px;
                background-color:#FFFFFF;
                font-size:13px;
                margin:10px auto;
                text-align:center;">
                <span style="font-size:18px;"><b><u>Option B:</u></b></span><br>
                <strong>Calculate LDL-cholesterol value from a non-fasting lipid profile using the Sampson equation</strong>
                <br><br>

                <a href="{CLINICAL_LINKS["SAMPSON_LDL_CALCULATOR"]}"
                   target="_blank"
                   class="qrisk-link"
                   style="
                       display:inline-block;
                       padding:10px 20px;
                       background-color:#0066cc;
                       color:white !important;
                       text-decoration:none;
                       border-radius:10px;
                       font-weight:bold;
                       font-size:15px;
                       animation: qriskPulse 1.8s ease-in-out infinite;">
                   <span style="color:white !important;">
                       Open Sampson Calculator
                   </span>
                </a>
            </div>
        '''),

        widgets.HTML('''
            <div class="oval-box" style="
                width:420px;
                box-sizing:border-box;
                padding:14px 18px;
                border:2px solid black;
                border-radius:25px;
                background-color:#FFFFFF;
                font-size:13px;
                margin:10px auto;
                text-align:center;">

                <span style="font-size:16px;">
                    <b><u><strong>Step 3: Enter LDL-C result</strong></u></b>
                </span>
                <br><br>

                <strong>Enter the LDL-cholesterol value obtained from either:</strong>
                <br>

                • A fasting lipid profile (Option A)<br>
                • The Sampson calculator (Option B)

                <br><br>

                <strong>This value will be used to determine the next recommended
                lipid-lowering therapy.</strong>
            </div>
        '''),

        ldl_input_box,
        ldl_result_container

    ], layout=widgets.Layout(align_items='center'))

    # ------------------------------------------------------------
    # Non-HDL follow-up section for LDL-C ≤ 2.6 branch
    # ------------------------------------------------------------
    non_hdl_label_new = widgets.HTML(
        value="<span style='font-size:16px; font-weight:bold;'>Enter Non-HDL result (mmol/L):</span>"
    )

    non_hdl_input_new = widgets.BoundedFloatText(
        value=0.0,
        min=0,
        max=20,
        step=0.1,
        description='',
        layout=widgets.Layout(width='120px')
    )

    non_hdl_button_new = widgets.Button(
        description="Enter",
        layout=widgets.Layout(width='100px', height='30px')
    )

    non_hdl_result_container = widgets.VBox(
        [],
        layout=widgets.Layout(align_items='center', width='100%')
    )

    non_hdl_input_box = widgets.HBox(
        [non_hdl_label_new, non_hdl_input_new, non_hdl_button_new],
        layout=widgets.Layout(
            justify_content='center',
            align_items='center',
            margin='10px 0'
        )
    )

    non_hdl_section = widgets.VBox([

        widgets.HTML('''
            <div class="oval-box" style="
                width:420px;
                box-sizing:border-box;
                padding:14px 18px;
                border:2px solid black;
                border-radius:25px;
                background-color:#FFFFFF;
                font-size:13px;
                margin:10px auto;
                text-align:center;">

                <span style="font-size:16px;">
                    <b><u><strong>Step 4: Enter follow-up Non-HDL result</strong></u></b>
                </span>
                <br><br>

                This is used after Ezetimibe monotherapy to decide whether to add Bempedoic Acid.
            </div>
        '''),

        non_hdl_input_box,
        non_hdl_result_container

    ], layout=widgets.Layout(align_items='center'))

    # ------------------------------------------------------------
    # Shared LDL-C required box
    # ------------------------------------------------------------


    q7_adherence_box = widgets.HTML('''
        <div style="font-size:14px;text-align:center;">
            Non-HDL cholesterol <strong>&gt;{NON_HDL_TREATMENT_THRESHOLDS["SECONDARY_TARGET"]} mmol/L</strong> after 3 months of confirmed adherence to therapy?
        </div>
    ''')

    ldl_required_box = widgets.HTML('''
        <div class="oval-box" style="
            width:420px;
            box-sizing:border-box;
            padding:14px 18px;
            border:2px solid black;
            border-radius:25px;
            background-color:#FFFFFF;
            font-size:13px;
            margin:10px auto;
            text-align:center;">
            <b>LDL-C value required to guide next lipid-lowering therapy</b>
        </div>
    ''')

    # ------------------------------------------------------------
    # Q8 branch if Q6 tolerance answer was No
    # ------------------------------------------------------------
    if q8_first:

        q8_info_button = widgets.ToggleButton(
            value=False,
            description="?",
            layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
        )

        q8_info_container = widgets.VBox(
            [],
            layout=widgets.Layout(align_items='center', width='100%')
        )

        def on_q8_info_toggle(change):
            if change['new']:
                q8_info_container.children = (
                    widgets.HTML('''
                        <div style="
                            border:1px solid #ccc;
                            padding:10px;
                            border-radius:10px;
                            background-color:#f9f9f9;
                            max-width:420px;
                            text-align:center;
                            font-size:13px;
                            line-height:1.7;">
                            <b>Statin intolerance definition:</b><br>
                            Statin intolerance is defined as the presence of clinically significant adverse effects
                            from statin therapy that are considered to represent an unacceptable risk to the patient,
                            or that may result in adherence to therapy being compromised.
                        </div>
                    '''),
                )
            else:
                q8_info_container.children = ()

        q8_info_button.observe(on_q8_info_toggle, names='value')

        q8_section = widgets.VBox([
            
            
                        
            widgets.HTML('''
                <div class="oval-box" style="
                    display: table;
                    max-width: 450px;
                    box-sizing: border-box;
                    padding:20px 35px;
                    border:2px solid black;
                    border-radius:25px;
                    background-color:#FFFFFF;
                    font-size:13px;
                    margin:10px auto;
                    text-align:center;
                    line-height:1.8;">
            
                    <span style="font-size:18px;">
                        <strong><u>Q8. Potential statin intolerance</u></strong>
                    </span>
                    <br><br>
            
                    <strong>Check AAC Statin Intolerance Algorithm for advice regarding management of potential statin intolerance.</strong>
                    <br><br>
            
                    <a href="{CLINICAL_LINKS["NEELI_STATIN_INTOLERANCE"]}"  
                       target="_blank"
                       rel="noopener noreferrer"
                       class="qrisk-link"
                       style="
                           display:inline-block;
                           padding:10px 20px;
                           background-color:#0066cc;
                           color:white !important;
                           text-decoration:none;
                           border-radius:10px;
                           font-weight:bold;
                           font-size:15px;
                           animation: qriskPulse 1.8s ease-in-out infinite;">
                       <span style="color:white !important;">
                           Open AAC Statin Intolerance Algorithm
                       </span>
                    </a>
            
                </div>
            '''),

            


            widgets.HBox([
                widgets.HTML('<div style="font-size:14px; font-weight:bold; margin-right:8px;">Definition of statin intolerance</div>'),
                q8_info_button
            ], layout=widgets.Layout(
                justify_content='center',
                align_items='center',
                margin='5px 0'
            )),

            q8_info_container,

            widgets.HTML('''
                <div class="oval-box" style="
                    display: table;
                    width: auto;
                    max-width: 540px;
                    box-sizing: border-box;
                    padding:20px 35px;
                    border:2px solid black;
                    border-radius:25px;
                    background-color:#d9fdd3;
                    font-size:13px;
                    margin:10px auto;
                    text-align:center;
                    line-height:1.8;">
                    <strong>All statin treatment not tolerated or contraindicated</strong>
                </div>
            ''')

        ], layout=widgets.Layout(align_items='center'))

        tolerance_branch_container = widgets.VBox([
            widgets.HTML('<div class="vertical-line"></div>'),
            q8_section,
            widgets.HTML('<div class="vertical-line"></div>'),
            ldl_required_box,
            sampson_section
        ], layout=widgets.Layout(align_items='center'))


    else:
    
        tolerance_branch_container = widgets.VBox([
            widgets.HTML('<div class="vertical-line"></div>'),
    
            q7_adherence_box,
    
            widgets.HTML('<div class="vertical-line"></div>'),
    
            ldl_required_box,
            sampson_section
    
        ], layout=widgets.Layout(align_items='center'))
        
    # ------------------------------------------------------------
    # LDL-C decision tree callback
    # ------------------------------------------------------------
    def on_ldl_button_click(b):

        ldl_val = ldl_input.value

        # Clear only the LDL-C result area.
        ldl_result_container.children = ()

        if ldl_val > LDL_C_TREATMENT_THRESHOLDS["PCSK9_MIN"]:

            ldl_result_container.children = (
                widgets.HTML(f'''
                    <div class="oval-box" style="
                        display: table;
                        width: auto;
                        max-width: 540px;
                        box-sizing: border-box;
                        padding:20px 35px;
                        border:2px solid black;
                        border-radius:25px;
                        background-color:#d9fdd3;
                        font-size:13px;
                        margin:10px auto;
                        text-align:center;
                        line-height:1.8;">
                        <span style="font-size:18px;"><strong><u>LDL-C &gt; {LDL_C_TREATMENT_THRESHOLDS["PCSK9_MIN"]} mmol/L:</u></strong></span><br>
                        <strong>• Offer PCSK9-inhibitor therapy</strong><br>
                        <strong>• Add Ezetimibe 10mg once daily if no contraindications</strong>
                    </div>
                '''),
            )

        elif ldl_val > LDL_C_TREATMENT_THRESHOLDS["CV_EVENT_PROMPT_MIN"]:

            cv_ldl_result_oval = widgets.HTML(f'''
                <div class="oval-box" style="
                    display: table;
                    width: auto;
                    max-width: 540px;
                    box-sizing: border-box;
                    padding:20px 35px;
                    border:2px solid black;
                    border-radius:25px;
                    background-color:#d9fdd3;
                    font-size:13px;
                    margin:10px auto;
                    text-align:center;
                    line-height:1.8;">

                    <span style="font-size:18px;">
                        <strong><u>LDL-C &gt; {LDL_C_TREATMENT_THRESHOLDS["CV_EVENT_PROMPT_MIN"]} mmol/L and ≤ {LDL_C_TREATMENT_THRESHOLDS["PCSK9_MIN"]} mmol/L:</u></strong>
                    </span>
                    <br>
                    <strong>Determine whether the patient meets “very high risk” criteria.</strong>
                </div>
            ''')

            cv_question = widgets.HTML('''
                <div style="
                    font-size:14px;
                    text-align:center;
                    max-width:720px;
                    line-height:1.8;
                    margin:0 auto;">
                    <div>Has the patient experienced recurrent CV events</div>
                </div>
            ''')

            cv_question_vertical_line = widgets.HTML('<div class="vertical-line"></div>')

            cv_yes_button = widgets.Button(
                description="Yes",
                layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
            )

            cv_no_button = widgets.Button(
                description="No",
                layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
            )

            cv_info_button = widgets.ToggleButton(
                value=False,
                description="?",
                layout=widgets.Layout(width='50px', height='50px', border_radius='50%')
            )

            cv_info_container = widgets.VBox(
                [],
                layout=widgets.Layout(align_items='center', width='100%')
            )

            cv_result_container = widgets.VBox(
                [],
                layout=widgets.Layout(align_items='center', width='100%')
            )

            def on_cv_info_toggle(change):
                if change['new']:
                    cv_info_container.children = (
                        widgets.HTML('''
                            <div style="
                                border:1px solid #ccc;
                                padding:10px;
                                border-radius:10px;
                                background-color:#f9f9f9;
                                max-width:520px;
                                text-align:center;
                                font-size:13px;">
                                <b>Recurrent CV events include:</b><br>
                                ACS, coronary or other arterial revascularisation procedures, CHD,
                                ischaemic stroke, and peripheral arterial disease.
                            </div>
                        '''),
                    )
                else:
                    cv_info_container.children = ()

            def on_cv_yes_clicked(b):
                highlight_button(
                    [cv_yes_button, cv_no_button, cv_info_button],
                    cv_yes_button
                )

                cv_result_container.children = (
                    widgets.HTML('''
                        <div class="oval-box" style="
                            display: table;
                            width: auto;
                            max-width: 540px;
                            box-sizing: border-box;
                            padding:20px 35px;
                            border:2px solid black;
                            border-radius:25px;
                            background-color:#d9fdd3;
                            font-size:13px;
                            margin:10px auto;
                            text-align:center;
                            line-height:1.8;">
                            <strong>• Offer PCSK9-inhibitor therapy</strong><br>
                            <strong>• Add Ezetimibe 10mg once daily if no contraindications</strong>
                        </div>
                    '''),
                )

            def on_cv_no_clicked(b):
                highlight_button(
                    [cv_yes_button, cv_no_button, cv_info_button],
                    cv_no_button
                )

                cv_result_container.children = (
                    widgets.HTML('''
                        <div class="oval-box" style="
                            display: table;
                            width: auto;
                            max-width: 540px;
                            box-sizing: border-box;
                            padding:20px 35px;
                            border:2px solid black;
                            border-radius:25px;
                            background-color:#d9fdd3;
                            font-size:13px;
                            margin:10px auto;
                            text-align:center;
                            line-height:1.8;">
                            <span style="font-size:14px;"><strong>• Offer Inclisiran therapy</strong></span><br><br>

                            <a href="{CLINICAL_LINKS["NICE_TA733_INCLISIRAN"]}"
                               target="_blank"
                               class="qrisk-link"
                               style="
                                   display:inline-block;
                                   padding:10px 20px;
                                   background-color:#0066cc;
                                   color:white !important;
                                   text-decoration:none;
                                   border-radius:10px;
                                   font-weight:bold;
                                   font-size:15px;
                                   animation: qriskPulse 1.8s ease-in-out infinite;">
                               <span style="color:white !important;">
                                   Open NICE TA733
                               </span>
                            </a>

                            <br><br>
                            <strong>• Add Ezetimibe 10mg once daily if no contraindications</strong>
                        </div>
                    '''),
                )

            cv_info_button.observe(on_cv_info_toggle, names='value')
            cv_yes_button.on_click(on_cv_yes_clicked)
            cv_no_button.on_click(on_cv_no_clicked)

            cv_buttons = widgets.HBox([
                cv_yes_button,
                widgets.HTML('<div class="horizontal-line"></div>'),
                cv_info_button,
                widgets.HTML('<div class="horizontal-line"></div>'),
                cv_no_button
            ], layout=widgets.Layout(
                justify_content='center',
                align_items='center',
                width='350px'
            ))

            cv_branch = widgets.VBox([
                widgets.HTML('<div class="vertical-line"></div>'),
                cv_ldl_result_oval,
                widgets.HTML('<div class="vertical-line"></div>'),
                cv_question,
                cv_question_vertical_line,
                cv_buttons,
                cv_info_container,
                cv_result_container
            ], layout=widgets.Layout(align_items='center'))

            ldl_result_container.children = (cv_branch,)

        elif ldl_val >= LDL_C_TREATMENT_THRESHOLDS["INCLISIRAN_MIN"]:

            ldl_result_container.children = (
                widgets.HTML(f'''
                    <div class="oval-box" style="
                        display: table;
                        width: auto;
                        max-width: 540px;
                        box-sizing: border-box;
                        padding:20px 35px;
                        border:2px solid black;
                        border-radius:25px;
                        background-color:#d9fdd3;
                        font-size:13px;
                        margin:10px auto;
                        text-align:center;
                        line-height:1.8;">
                        <span style="font-size:18px;"><strong><u>LDL-C ≥ {LDL_C_TREATMENT_THRESHOLDS["INCLISIRAN_MIN"]} mmol/L and ≤ {LDL_C_TREATMENT_THRESHOLDS["CV_EVENT_PROMPT_MIN"]} mmol/L:</u></strong></span><br>
                        <span style="font-size:14px;"><strong>• Offer Inclisiran therapy</strong></span>
                        <br><br>

                        <a href="{CLINICAL_LINKS["NICE_TA733_INCLISIRAN"]}"
                           target="_blank"
                           class="qrisk-link"
                           style="
                               display:inline-block;
                               padding:10px 20px;
                               background-color:#0066cc;
                               color:white !important;
                               text-decoration:none;
                               border-radius:10px;
                               font-weight:bold;
                               font-size:15px;
                               animation: qriskPulse 1.8s ease-in-out infinite;">
                           <span style="color:white !important;">
                               Open NICE TA733
                           </span>
                        </a>

                        <br><br>
                        <span style="font-size:14px;"><strong>• Add Ezetimibe 10mg once daily if no contraindications</strong></span>
                    </div>
                '''),
            )

        else:

            ldl_result_container.children = (

                widgets.HTML(f'''
                    <div class="oval-box" style="
                        display: table;
                        width: auto;
                        max-width: 540px;
                        box-sizing: border-box;
                        padding:20px 35px;
                        border:2px solid black;
                        border-radius:25px;
                        background-color:#d9fdd3;
                        font-size:13px;
                        margin:10px auto;
                        text-align:center;
                        line-height:1.8;">
                        <span style="font-size:18px;"><strong><u>LDL-C &lt; {LDL_C_TREATMENT_THRESHOLDS["INCLISIRAN_MIN"]} mmol/L:</u></strong></span><br>
                        <strong>• Consider Ezetimibe 10mg monotherapy.</strong><br>
                        <strong>• Check non-HDL after 3 months.</strong>
                    </div>
                '''),
                non_hdl_section
            )

    ldl_button.on_click(on_ldl_button_click)

    # ------------------------------------------------------------
    # Non-HDL result callback
    # ------------------------------------------------------------
    def on_non_hdl_button_new_click(b):

        nh_val = non_hdl_input_new.value

        if nh_val > NON_HDL_TREATMENT_THRESHOLDS["SECONDARY_TARGET"]:
            non_hdl_result_container.children = (
                widgets.HTML(f'''
                    <div class="oval-box" style="
                        display: table;
                        width: auto;
                        max-width: 520px;
                        box-sizing: border-box;
                        padding:14px 25px;
                        border:2px solid black;
                        border-radius:25px;
                        background-color:#d9fdd3;
                        font-size:13px;
                        margin:10px auto;
                        text-align:center;
                        line-height:1.8;">
                        <span style="font-size:16px;"><strong><u>Non-HDL &gt; {NON_HDL_TREATMENT_THRESHOLDS["SECONDARY_TARGET"]} mmol/L:</u></strong></span><br>
                        <strong>Add Bempedoic Acid 180mg daily if no contraindications.</strong>
                    </div>
                '''),
            )
        else:
            non_hdl_result_container.children = (
                widgets.HTML(f'''
                    <div class="oval-box" style="
                        display: table;
                        width: auto;
                        max-width: 520px;
                        box-sizing: border-box;
                        padding:14px 25px;
                        border:2px solid black;
                        border-radius:25px;
                        background-color:#d9fdd3;
                        font-size:13px;
                        margin:10px auto;
                        text-align:center;
                        line-height:1.8;">
                        <span style="font-size:16px;"><strong><u>Non-HDL ≤ {NON_HDL_TREATMENT_THRESHOLDS["SECONDARY_TARGET"]} mmol/L:</u></strong></span><br>
                        <strong>Continue Ezetimibe monotherapy.</strong>
                    </div>
                '''),
            )

    non_hdl_button_new.on_click(on_non_hdl_button_new_click)

    # ------------------------------------------------------------
    # Add the completed branch to the visible flowchart
    # ------------------------------------------------------------
    new_branch = widgets.VBox(
        [tolerance_branch_container],
        layout=widgets.Layout(align_items='center')
    )

    atorvastatin_flowchart_container.children = (
        atorvastatin_flowchart_container.children + (new_branch,)
    )

    state['atorvastatin_tolerance_branch_created'] = True

def on_atorvastatin_yes_button_click(b):
    # Q5 Yes: already on Atorvastatin 80mg daily
    state['atorvastatin_choice'] = 'yes'

    highlight_button(
        [atorvastatin_yes_button, atorvastatin_no_button, atorvastatin_info_button],
        atorvastatin_yes_button
    )

    # Rebuild this branch from scratch, so switching Yes/No does not create a chain
    update_atorvastatin_branch()


def on_atorvastatin_no_button_click(b):
    # Q5 No: not already on Atorvastatin 80mg daily
    state['atorvastatin_choice'] = 'no'

    highlight_button(
        [atorvastatin_yes_button, atorvastatin_no_button, atorvastatin_info_button],
        atorvastatin_no_button
    )

    # Rebuild this branch from scratch, so switching Yes/No does not create a chain
    update_atorvastatin_branch()


def on_atorvastatin_tolerance_no_button_click(b):
    # Q6 No: not tolerating Atorvastatin 80mg daily
    highlight_button(
        [
            atorvastatin_tolerance_yes_button,
            atorvastatin_tolerance_no_button,
            atorvastatin_tolerance_info_button
        ],
        atorvastatin_tolerance_no_button
    )

    # Q6 No leads to Q8 first, then to the same LDL-C/Sampson pathway as Q7.
    on_atorvastatin_tolerance_yes_button_click(
        b=None,
        q8_first=True
    )


# Connect ONLY the tolerance buttons here.
# The main Atorvastatin Yes/No buttons are connected in your global event-handler cell.
atorvastatin_tolerance_yes_button.on_click(on_atorvastatin_tolerance_yes_button_click)
atorvastatin_tolerance_no_button.on_click(on_atorvastatin_tolerance_no_button_click)





def toggle_atorvastatin_info_box(b):
    # Example info box toggle – customize as required.
    print("Toggle Atorvastatin info box")
    # You can implement a similar toggle behavior as for your other info buttons.

<!-- ## 20. Atorvastatin tolerance and LDL-C treatment logic

Handles atorvastatin tolerance, LDL-C threshold routing, PCSK9/Ezetimibe/Inclisiran prompts, and Non-HDL follow-up. -->

<!-- ## 21. Standalone cholesterol validation widgets

Preserves the standalone cholesterol type dropdown, numeric input, and submit button definitions. -->

In [21]:
# Create selection dropdown for cholesterol type
cholesterol_dropdown = widgets.Dropdown(
    options=[("Total Cholesterol (TC)", "TC"), 
             ("LDL-C", "LDL"), 
             ("Non-HDL", "Non-HDL")],
    description="Select Type:",
    layout=widgets.Layout(width='300px')
)

# Create input box for user to enter cholesterol value
cholesterol_input = widgets.BoundedFloatText(
    min=0,
    description="Enter Value (mmol/L):",
    layout=widgets.Layout(width='300px')
)

# Create a button to validate cholesterol level
submit_button = widgets.Button(description="Submit")











# Update the restore_state function to ensure ACS oval visibility is updated


<!-- ## 22. Restore-state logic

Rebuilds the visible branch from the state dictionary after selections change. -->

In [22]:
def restore_state():
    # Restore the previous state of the flowchart based on the saved state
    if state['current_state'] == 'primary_prevention':
        update_primary_flowchart()
    elif state['current_state'] == 'no_button':
        on_no_button_click(None, restore=True)
    elif state['current_state'] == 'yes_button':
        on_yes_button_click(None, restore=True)
    elif state['current_state'] == 'age_option_1':
        select_age_option_84(None, restore=True)
    elif state['current_state'] == 'age_option_2':
        select_age_option_above_84(None, restore=True)
    elif state['current_state'] == 'ckd_no_button':
        on_ckd_no_button_click(None, restore=True)
    elif state['current_state'] == 'ckd_yes_button':
        on_ckd_yes_button_click(None, restore=True)
    elif state['current_state'] == 'diabetes_no_button':
        on_diabetes_no_button_click(None, restore=True)
    elif state['current_state'] == 'diabetes_yes_button':
        on_diabetes_yes_button_click(None, restore=True)
    elif state['current_state'] == 'type1_selected':
        on_type1_button_click(None, restore=True)
    elif state['current_state'] == 'type2_selected':
        on_type2_button_click(None, restore=True)
    elif state['current_state'] == 'secondary_prevention':
        update_secondary_flowchart()
    elif state['current_state'] == 'secondary_yes_button':
        on_secondary_yes_button_click(None, restore=True)
        if state.get('non_hdl_flowchart_created', False):
            create_non_hdl_flowchart()



    
    elif state['current_state'] == 'acs_yes_button':
        # Ensure the ACS Yes state, including highlighting the button and showing the oval
        on_acs_yes_button_click(None, restore=True)
    elif state['current_state'] == 'acs_no_button':
        on_acs_no_button_click(None, restore=True)
        
    elif state.get('statin_oval_box_visible', False):
        show_statin_oval_box()

    # elif state['current_state'] == 'drug_interaction_yes_button':
    #     on_drug_interaction_yes_button_click(None, restore=True)
    # elif state['current_state'] == 'drug_interaction_no_button':
    #     on_drug_interaction_no_button_click(None, restore=True)
    # # Also restore the info box visibility
    # elif state['drug_interaction_info_box_visible']:
    #     toggle_drug_interaction_info_box(None)
    # # After restoring, update the oval visibility if needed



    elif state['current_state'] == 'triglycerides_yes_button':
        on_triglycerides_yes_button_click(None, restore=True)
    elif state['current_state'] == 'triglycerides_no_button':
        on_triglycerides_no_button_click(None, restore=True)

    # Restore Oval Boxes
    elif state.get('statin_oval_box_visible', False):
        show_statin_oval_box()
    else:
        hide_statin_oval_box()
    if state.get('acs_oval_box_visible', False):
        show_acs_oval_box()
    else:
        hide_acs_oval_box()
    if state.get('modifiable_risk_factors_oval_box_visible', False):
        show_modifiable_risk_factors_oval_box()
    else:
        hide_modifiable_risk_factors_oval_box()

    # Restore drug interaction flowchart
    if state.get('drug_interaction_flowchart_created', False):
        create_drug_interaction_flowchart()
        if state.get('drug_interaction_yes_button', False):
            on_drug_interaction_yes_button_click(None, restore=True)
        elif state.get('drug_interaction_no_button', False):
            on_drug_interaction_no_button_click(None, restore=True)

    # Restore Non-HDL flowchart
    if state.get('non_hdl_flowchart_created', False):
        create_non_hdl_flowchart()
        if state.get('non_hdl_yes_button', False):
            on_non_hdl_yes_button_click(None, restore=True)
        elif state.get('non_hdl_no_button', False):
            on_non_hdl_no_button_click(None, restore=True)
    else:
        clear_non_hdl_flowchart()

    # Restore Triglycerides flowchart state if Non-HDL "Yes" is selected
    if state.get('triglycerides_flowchart_created', False):
        create_triglycerides_flowchart()
        if state.get('triglycerides_yes_button', False):
            on_triglycerides_yes_button_click(None, restore=True)
        elif state.get('triglycerides_no_button', False):
            on_triglycerides_no_button_click(None, restore=True)


    state['fh_line_visible'] = False
        


<!-- ## 23. Event-handler wiring and final initialisation

Attaches callbacks to all buttons and calls restore_state() to initialise the UI. -->

In [23]:
# Assign the event handlers to the buttons
primary_button.on_click(on_primary_button_click)
no_button.on_click(on_no_button_click)
yes_button.on_click(on_yes_button_click)
ckd_no_button.on_click(on_ckd_no_button_click)
ckd_yes_button.on_click(on_ckd_yes_button_click)
diabetes_no_button.on_click(on_diabetes_no_button_click)
diabetes_yes_button.on_click(on_diabetes_yes_button_click)
type1_button.on_click(on_type1_button_click)
type2_button.on_click(on_type2_button_click)
hyperlipidaemia_button.on_click(on_hyperlipidaemia_button_click)
secondary_button.on_click(on_secondary_button_click)
info_button.on_click(toggle_info_box)
ckd_info_button.on_click(toggle_ckd_info_box)
diabetes_info_button.on_click(toggle_diabetes_info_box)

secondary_yes_button.on_click(on_secondary_yes_button_click)
secondary_no_button.on_click(on_secondary_no_button_click)
secondary_info_button.on_click(on_secondary_info_button_click)
drug_interaction_yes_button.on_click(on_drug_interaction_yes_button_click)
drug_interaction_no_button.on_click(on_drug_interaction_no_button_click)

acs_yes_button.on_click(on_acs_yes_button_click)
acs_no_button.on_click(on_acs_no_button_click)
acs_info_button.on_click(toggle_acs_info_box)


# Assign event handlers to Non-HDL buttons
non_hdl_yes_button.on_click(on_non_hdl_yes_button_click)
non_hdl_no_button.on_click(on_non_hdl_no_button_click)
non_hdl_info_button.on_click(toggle_non_hdl_info_box)


# Add event handlers for the Triglycerides buttons
triglycerides_yes_button.on_click(on_triglycerides_yes_button_click)
triglycerides_no_button.on_click(on_triglycerides_no_button_click)
triglycerides_info_button.on_click(toggle_triglycerides_info_box)

ldl_yes_button.on_click(on_ldl_yes_button_click)
ldl_no_button.on_click(on_ldl_no_button_click)

atorvastatin_yes_button.on_click(on_atorvastatin_yes_button_click)
atorvastatin_no_button.on_click(on_atorvastatin_no_button_click)
atorvastatin_info_button.on_click(toggle_atorvastatin_info_box)


restore_state()
